# PhysioGraph extension: circulatory context of SpO2 instability

Locked 2026-09-05 UTC before extraction of the new joint-signal features.
Scope: **MIMIC and eICU only**, original files under My Drive/Data. This develops
the current SpO2-instability project, especially its cross-database myocardial
injury association. No molecular or unrelated datasets are part of this work.

## Biological question

Does an abrupt SpO2 change accompanied by a contemporaneous fall in systemic
arterial pressure distinguish episodes followed by myocardial injury and
death from isolated SpO2 changes? This tests a circulatory-context hypothesis;
it does not assume that a peripheral oximeter measures cardiac output or
microcirculation. The prior analysis already found that recovery rises can
predict events, so absolute changes remain primary and direction is explicit.

## Unchanged anchors

Use the production PhysioGraph adult, explicit-HF phenotype, first ICU per
hospital encounter, early ICU admission or cardiogenic shock, and exclusion
of death or end of observable follow-up by four hours. Follow-up ends at
the earlier of ICU and hospital discharge, matching production code.
Use 0-4-hour SpO2 values in 50-100%,
same-time medians, 15-minute median bins, at least three bins and two
forward transitions no more than 30 minutes apart. The SpO2 exposure remains
an absolute adjacent change of at least four percentage points.

## New feature and comparators

For each qualifying SpO2 transition, obtain MAP medians in its two bins.
Never carry a pressure measurement across an empty bin. A circulatory decline
is a MAP decrease of at least 10 mmHg over that same transition. MAP values
outside 20-200 mmHg are excluded; the threshold is fixed before outcomes.

Classify eligible encounters into four states: neither event, SpO2 change
only, MAP decline only, both. Also record whether the events occur on the
SAME transition; co-occurrence anywhere within four hours is not synchrony.
Require at least two jointly observed transitions. Do not label missing MAP
as stable circulation. Preserve arterial and noninvasive pressure provenance;
eICU currently lacks the noninvasive-pressure source.

Primary question: does synchronous instability carry information beyond
separate SpO2-jump and MAP-decline indicators, their event burdens, absolute
levels, age, sex, heart rate and sampling? Secondary fixed checks: downward
versus upward SpO2 jumps; stable absolute oxygenation (all bins >=90%);
normal measured baseline lactate (<2 mmol/L); invasive-pressure-only MIMIC.
Normal lactate requires an actual measurement; missing is not normal.

## Endpoints and safeguards

Two primary endpoint families: subsequent hospital death and the existing
assay-matched troponin rise endpoint after the four-hour landmark. For the
first feasibility/feature build, extract raw labs through 28 hours and report
coverage before any effect estimates. Freeze the exact window and model
after verifying assay/time support, before new feature-outcome associations.
This document freezes the feature, not a fully specified final model.

An eventual mortality model must compare against separate component signals
and baseline physiology/respiratory treatment, with patient-clustered
uncertainty and eICU hospital heterogeneity. No future lactate, troponin,
ventilation or pressors enter baseline adjustment. Myocardial injury is not
adjudicated infarction. Troponin ordering requires an observation analysis.
Do not claim full SCAI equivalence when source fields are missing.

Minimum support for a primary fitted contrast: 100 endpoint events overall,
at least 20 events and 20 non-events in the synchronous group and comparator,
and adequate model/weight support. Sparse strata remain descriptive. Do not
choose a replacement threshold or endpoint to rescue a failed result. Report
both databases separately, transfer definitions without retuning, and retain
negative results. Previously explored databases are not pristine holdouts.

## Gates for advancing

A robust cross-database association is only the first gate. Useful early
warning also requires improvement over component signals, lactate and
faithfully available shock measures; calibration/net benefit; temporal and
treatment controls; and direct mechanistic corroboration. Mortality reduction
needs intervention evidence. Neither association nor a new score alone
completes the user's paradigm-shifting discovery goal.

In [1]:
CONTEXT_PROTOCOL_SHA256='f7d78bdc2aba420320434e9a72a576b72dfd825353491f0a36d1569467b35b23'

In [2]:
import sys, importlib.util, subprocess, json, hashlib, os
from pathlib import Path
from datetime import datetime, timezone
requirements={'numpy':'numpy==2.2.6','pandas':'pandas==2.2.3','duckdb':'duckdb==1.4.3','yaml':'pyyaml'}
missing=[p for m,p in requirements.items() if importlib.util.find_spec(m) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import numpy as np
import pandas as pd
import duckdb
IN_COLAB=importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE=Path('/content/drive/MyDrive')
else:DRIVE=Path.home()/'Library/CloudStorage/GoogleDrive-2arnavmana@gmail.com/My Drive'
PROJECT=DRIVE/'Projects'/'PhysioGraph'
assert (PROJECT/'src/physiograph').is_dir(), 'Keep the PhysioGraph project beside Data in Drive'
sys.path.insert(0,str(PROJECT/'src'))
from physiograph.etl.audit import AuditLogger
from physiograph.etl.mimic_extractor import _load_mimic_cohort, _mimic_chartevent_item_map, _mimic_lab_item_map
from physiograph.etl.eicu_extractor import _load_eicu_patients, _build_eicu_diagnosis_flags, _build_eicu_cohort
from physiograph.constants import EICU_LAB_NAME_MAP, EICU_VITAL_COLUMN_MAP
PRIVATE=DRIVE/'Data'/'PhysioGraph_Biological_Discovery_20260905'
PRIVATE.mkdir(exist_ok=True)
OUT=PROJECT/'research'/'spo2_circulatory_context';OUT.mkdir(parents=True,exist_ok=True)
ENVIRONMENT='google_colab' if IN_COLAB else 'local'
def context_json(name,value):
    (OUT/name).write_text(json.dumps(value,indent=2,allow_nan=False,default=str)+'\n')
def context_status(stage,state='running'):
    context_json('run_status.json',dict(stage=stage,status=state,environment=ENVIRONMENT,
        updated_utc=datetime.now(timezone.utc).isoformat(),goal_complete=False))
    print(stage,state,flush=True)
context_json('protocol.json',dict(sha256=CONTEXT_PROTOCOL_SHA256,frozen_before_feature_extraction=True))
con=duckdb.connect(str(PRIVATE/'spo2_context.duckdb'))
con.execute('SET threads=2');con.execute("SET memory_limit='2GB'")
RAW={d:DRIVE/'Data'/d/'Full' for d in ['MIMIC','eICU']}
sources={}
def qs(value):return "'"+str(value).replace("'","''")+"'"
def raw_csv(dataset,name):
    p=RAW[dataset]/(name+'.csv');s=p.stat()
    sources[dataset+'/'+name]=dict(path=str(p),bytes=s.st_size,mtime_ns=s.st_mtime_ns)
    return f'read_csv({qs(p)},header=true,all_varchar=true,sample_size=10000,strict_mode=true)'
context_status('production cohort reconstruction')
cohorts={};cohort_summary=[]
for d,source in [('eicu','eICU'),('mimic','MIMIC')]:
    audit=AuditLogger(d)
    if d=='eicu' and globals().get('RESUME_EICU',False):
        # Explicit recovery of the successfully committed step from this run.
        c=con.execute('SELECT * FROM eicu_cohort').df()
        assert c.stay_id.is_unique and c.age.ge(18).all() and c.cohort_hf_flag.eq(1).all()
        assert (c.early_icu_flag.eq(1)|c.shock_icd_flag.eq(1)).all()
        print('Resumed committed eICU cohort from the preceding local attempt:',len(c))
        for name in ['patient','diagnosis','admissionDx']:raw_csv(source,name)
    elif d=='eicu':
        patient=_load_eicu_patients(RAW[source],audit)
        flags=_build_eicu_diagnosis_flags(RAW[source],patient)
        c,_=_build_eicu_cohort(RAW[source],patient,flags,audit,max_stays=None)
        outcome=patient.set_index('patientunitstayid').hospitaldischargestatus.str.lower()
        c['hospital_mortality']=c.stay_id.map(outcome.map({'expired':1.,'alive':0.}))
        for name in ['patient','diagnosis','admissionDx']:raw_csv(source,name)
    else:
        c,_,full=_load_mimic_cohort(RAW[source],audit,max_stays=None)
        mortality=pd.read_csv(RAW[source]/'admissions.csv',usecols=['hadm_id','hospital_expire_flag']).set_index('hadm_id').hospital_expire_flag
        c['hospital_mortality']=c.hadm_id.map(mortality).astype(float)
        for name in ['patients','admissions','icustays','diagnoses_icd']:raw_csv(source,name)
    before=len(c)
    exclude=c.death_offset_minutes.between(0,240)|c.followup_end_offset_minutes.le(240)
    c=c.loc[~exclude].copy()
    assert c.stay_id.is_unique
    c['encounter_id']=c['hadm_id'] if d=='mimic' else c['hospital_encounter_id']
    con.register('cohort_frame',c)
    con.execute(f'CREATE OR REPLACE TABLE {d}_cohort AS SELECT * FROM cohort_frame')
    cohorts[d]=c
    cohort_summary.append(dict(dataset=d,before_exclusion=before,eligible=len(c),
        people=c.person_id.nunique(),known_hospital_outcomes=c.hospital_mortality.notna().sum()))
context_json('cohort_support.json',cohort_summary)
print(pd.DataFrame(cohort_summary).to_string(index=False))

production cohort reconstruction running
dataset  before_exclusion  eligible  people  known_hospital_outcomes
   eicu             12304     12028   10647                    11910
  mimic             17892     17758   13464                    17758


In [3]:
context_status('eICU raw vital scan')
source=raw_csv('eICU','vitalPeriodic')
concepts={'sao2':'spo2','systemicmean':'map','heartrate':'hr','systemicsystolic':'sbp','respiration':'resp_rate'}
con.execute(f"""CREATE OR REPLACE TABLE eicu_vitals_wide AS
SELECT try_cast(v.patientunitstayid AS BIGINT) stay_id,
try_cast(v.observationoffset AS DOUBLE) event_minute,
{','.join('try_cast(v.'+col+' AS DOUBLE) AS '+concept for col,concept in concepts.items())}
FROM {source} v JOIN eicu_cohort c ON try_cast(v.patientunitstayid AS BIGINT)=c.stay_id
WHERE try_cast(v.observationoffset AS DOUBLE)>=0 AND try_cast(v.observationoffset AS DOUBLE)<240""")
con.execute("CREATE OR REPLACE TABLE eicu_vitals AS "+' UNION ALL '.join(
    f"SELECT stay_id,event_minute,{qs(concept)} concept,{concept} AS value,'periodic_invasive_context' source_type FROM eicu_vitals_wide WHERE {concept} IS NOT NULL"
    for concept in concepts.values()))
context_status('MIMIC raw vital scan')
itemmap=_mimic_chartevent_item_map(RAW['MIMIC'])
chosen={i:m for i,m in itemmap.items() if m in ['spo2','map','sbp','hr','resp_rate','fio2']}
mapping=pd.DataFrame([dict(itemid=i,concept=m) for i,m in chosen.items()]);con.register('vital_map',mapping)
source=raw_csv('MIMIC','chartevents');raw_csv('MIMIC','d_items')
con.execute(f"""CREATE OR REPLACE TABLE mimic_vitals AS
SELECT c.stay_id,date_diff('second',c.admit_time,try_cast(v.charttime AS TIMESTAMP))/60.0 event_minute,
m.concept,try_cast(v.valuenum AS DOUBLE) AS value,
CASE WHEN try_cast(v.itemid AS BIGINT) IN (220181,220179) THEN 'noninvasive'
WHEN m.concept IN ('map','sbp') THEN 'invasive' ELSE 'charted' END source_type
FROM {source} v JOIN mimic_cohort c ON try_cast(v.stay_id AS BIGINT)=c.stay_id
JOIN vital_map m ON try_cast(v.itemid AS BIGINT)=m.itemid
WHERE try_cast(v.charttime AS TIMESTAMP)>=c.admit_time AND try_cast(v.charttime AS TIMESTAMP)<c.admit_time+INTERVAL '4 hours'
AND coalesce(try_cast(v.warning AS DOUBLE),0)<>1
AND try_cast(v.valuenum AS DOUBLE) IS NOT NULL""")
con.execute('CREATE OR REPLACE TABLE mimic_context_quality_marker AS SELECT 1 AS version')
context_status('joint SpO2 and MAP feature construction')
coverage=[]
for d in ['eicu','mimic']:
    con.execute(f"""CREATE OR REPLACE TABLE {d}_bins AS
    WITH dedup AS (
      SELECT stay_id,event_minute,concept,median(value) AS value FROM {d}_vitals
      WHERE (concept='spo2' AND value BETWEEN 50 AND 100)
         OR (concept='map' AND value BETWEEN 20 AND 200)
         OR (concept='hr' AND value BETWEEN 20 AND 250)
         OR (concept='sbp' AND value BETWEEN 30 AND 300)
         OR (concept='resp_rate' AND value BETWEEN 2 AND 80)
      GROUP BY stay_id,event_minute,concept
    ) SELECT stay_id,floor(event_minute/15)::INTEGER bin,concept,median(value) AS value,
      median(event_minute) AS representative_minute,
      count(*) n_readings FROM dedup GROUP BY stay_id,bin,concept""")
    bins=con.execute(f'SELECT * FROM {d}_bins ORDER BY stay_id,bin').df()
    wide=bins.pivot(index=['stay_id','bin'],columns='concept',values='value').reset_index()
    clock=bins[bins.concept.eq('spo2')][['stay_id','bin','representative_minute']]
    wide=wide.merge(clock,on=['stay_id','bin'],how='left',validate='one_to_one')
    feature_rows=[]
    for stay,s in wide.groupby('stay_id',sort=False):
        s=s[s.spo2.notna()].sort_values('bin')
        gap=s.representative_minute.diff();ds=s.spo2.diff();dm=s['map'].diff()
        transition=gap.gt(0)&gap.le(30)
        joint=transition&dm.notna()
        jump=ds.abs().ge(4)&joint;fall=dm.le(-10)&joint
        joint_n=int(joint.sum());spo2_n=len(s)
        row=dict(stay_id=int(stay),spo2_bins=spo2_n,spo2_transitions=int(transition.sum()),
            joint_transitions=joint_n,joint_eligible=spo2_n>=3 and joint_n>=2,
            spo2_jump_any=int((ds.abs().ge(4)&transition).any()),
            joint_spo2_jump_any=int(jump.any()),map_fall_any=int(fall.any()),
            synchronous_any=int((jump&fall).any()),synchronous_drop=int((jump&fall&ds.lt(0)).any()),
            synchronous_rise=int((jump&fall&ds.gt(0)).any()),
            joint_spo2_jumps=int(jump.sum()),map_falls=int(fall.sum()),
            spo2_mean=float(s.spo2.mean()),spo2_min=float(s.spo2.min()),
            map_mean=float(s['map'].mean()),map_min=float(s['map'].min()),
            hr_mean=float(s.hr.mean()),map_observed_bins=int(s['map'].notna().sum()))
        feature_rows.append(row)
    features=pd.DataFrame(feature_rows)
    con.register('feature_frame',features)
    con.execute(f'CREATE OR REPLACE TABLE {d}_features AS SELECT * FROM feature_frame')
    eligible=features[features.joint_eligible]
    coverage.append(dict(dataset=d,cohort=len(cohorts[d]),spo2_observed=len(features),
        spo2_dynamics=int(((features.spo2_bins>=3)&(features.spo2_transitions>=2)).sum()),
        jointly_observed=len(eligible),synchronous=int(eligible.synchronous_any.sum()),
        simultaneous_drop=int(eligible.synchronous_drop.sum()),
        simultaneous_rise=int(eligible.synchronous_rise.sum())))
context_json('joint_feature_support.json',coverage)
print(pd.DataFrame(coverage).to_string(index=False))
# No feature/outcome association is computed in this feasibility stage.
context_json('source_metadata.json',sources)
context_status('cohort and joint-feature feasibility','complete')
con.close()

eICU raw vital scan running
MIMIC raw vital scan running
joint SpO2 and MAP feature construction running
dataset  cohort  spo2_observed  spo2_dynamics  jointly_observed  synchronous  simultaneous_drop  simultaneous_rise
   eicu   12028          11730          11452              1022           76                 36                 43
  mimic   17758          17172           4711              2846          223                129                106
cohort and joint-feature feasibility complete


In [4]:
# This cell can resume from the completed raw vital/cohort extraction.
# It does not regenerate or alter the locked SpO2/MAP exposure.
import sys, json, hashlib, importlib.util
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import duckdb
IN_COLAB=importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
if IN_COLAB:DRIVE=Path('/content/drive/MyDrive')
else:DRIVE=Path.home()/'Library/CloudStorage/GoogleDrive-2arnavmana@gmail.com/My Drive'
PROJECT=DRIVE/'Projects'/'PhysioGraph'
sys.path.insert(0,str(PROJECT/'src'))
from physiograph.etl.mimic_extractor import _mimic_lab_item_map
from physiograph.constants import EICU_LAB_NAME_MAP
PRIVATE=DRIVE/'Data'/'PhysioGraph_Biological_Discovery_20260905'
OUT=PROJECT/'research'/'spo2_circulatory_context'
RAW={d:DRIVE/'Data'/d/'Full' for d in ['MIMIC','eICU']}
con=duckdb.connect(str(PRIVATE/'spo2_context.duckdb'))
con.execute('SET threads=2');con.execute("SET memory_limit='2GB'")
sources=json.loads((OUT/'source_metadata.json').read_text())
def context_json(name,value):
    (OUT/name).write_text(json.dumps(value,indent=2,allow_nan=False,default=str)+'\n')
def context_status(stage,state='running'):
    context_json('run_status.json',dict(stage=stage,status=state,
        environment='google_colab' if IN_COLAB else 'local',
        updated_utc=datetime.now(timezone.utc).isoformat(),goal_complete=False))
    print(stage,state,flush=True)
def qs(value):return "'"+str(value).replace("'","''")+"'"
def raw_csv(dataset,name):
    p=RAW[dataset]/(name+'.csv');s=p.stat()
    sources[dataset+'/'+name]=dict(path=str(p),bytes=s.st_size,mtime_ns=s.st_mtime_ns)
    return f'read_csv({qs(p)},header=true,all_varchar=true,sample_size=10000,strict_mode=true)'
context_status('eICU myocardial-injury and baseline lab extraction')
chosen={k:v for k,v in EICU_LAB_NAME_MAP.items() if v in ['troponin_i','troponin_t','lactate','creatinine','ph']}
mapping=pd.DataFrame([dict(raw_name=k,concept=v) for k,v in chosen.items()]);con.register('lab_map',mapping)
source=raw_csv('eICU','lab')
con.execute(f"""CREATE OR REPLACE TABLE eicu_context_labs AS
SELECT c.stay_id,try_cast(l.labresultoffset AS DOUBLE) event_minute,
try_cast(l.labresultrevisedoffset AS DOUBLE) available_minute,
m.concept,l.labname raw_name,lower(trim(coalesce(l.labmeasurenamesystem,l.labmeasurenameinterface,''))) unit,
try_cast(l.labresult AS DOUBLE) AS value
FROM {source} l JOIN eicu_cohort c ON try_cast(l.patientunitstayid AS BIGINT)=c.stay_id
JOIN lab_map m ON lower(trim(l.labname))=m.raw_name
WHERE try_cast(l.labresultoffset AS DOUBLE) BETWEEN 0 AND least(1680,c.followup_end_offset_minutes)
AND try_cast(l.labresult AS DOUBLE) IS NOT NULL""")
context_status('MIMIC myocardial-injury and baseline lab extraction')
chosen={k:v for k,v in _mimic_lab_item_map(RAW['MIMIC']).items() if v in ['troponin_i','troponin_t','lactate','creatinine','ph']}
mapping=pd.DataFrame([dict(itemid=k,concept=v) for k,v in chosen.items()]);con.register('lab_map',mapping)
source=raw_csv('MIMIC','labevents');raw_csv('MIMIC','d_labitems')
con.execute(f"""CREATE OR REPLACE TABLE mimic_context_labs AS
SELECT c.stay_id,date_diff('second',c.admit_time,try_cast(l.charttime AS TIMESTAMP))/60.0 event_minute,
date_diff('second',c.admit_time,try_cast(l.storetime AS TIMESTAMP))/60.0 available_minute,
m.concept,l.itemid raw_name,lower(trim(coalesce(l.valueuom,''))) unit,try_cast(l.valuenum AS DOUBLE) AS value
FROM {source} l JOIN mimic_cohort c ON try_cast(l.hadm_id AS BIGINT)=c.hadm_id
JOIN lab_map m ON try_cast(l.itemid AS BIGINT)=m.itemid
WHERE try_cast(l.charttime AS TIMESTAMP)>=c.admit_time
AND date_diff('second',c.admit_time,try_cast(l.charttime AS TIMESTAMP))/60.0<=least(1680,c.followup_end_offset_minutes)
AND try_cast(l.valuenum AS DOUBLE) IS NOT NULL""")
support=[];unit_rows=[]
for d in ['eicu','mimic']:
    c=con.execute(f'SELECT * FROM {d}_cohort').df().set_index('stay_id')
    labs=con.execute(f'SELECT * FROM {d}_context_labs').df()
    bounds={'troponin_t':(0,1000000),'troponin_i':(0,1000000),'lactate':(.1,40),'creatinine':(.1,30),'ph':(6.5,8)}
    keep=np.zeros(len(labs),dtype=bool)
    for concept,(lo,hi) in bounds.items():keep|=labs.concept.eq(concept)&labs.value.between(lo,hi)
    labs=labs[keep].copy()
    unit_rows.extend(labs.groupby(['concept','raw_name','unit']).size().rename('records').reset_index().assign(dataset=d).to_dict('records'))
    labs=labs.groupby(['stay_id','event_minute','concept','unit'],as_index=False).agg(value=('value','median'),available_minute=('available_minute','max'))
    base=labs[labs.event_minute.le(240)].sort_values('event_minute')
    for marker in ['lactate','creatinine','ph']:
        last=base[base.concept.eq(marker)].drop_duplicates('stay_id',keep='last').set_index('stay_id')
        c['baseline_'+marker]=last.value
        c['baseline_'+marker+'_available_minute']=last.available_minute
    base_t=base[base.concept.str.startswith('troponin')&base.value.gt(0)].drop_duplicates(['stay_id','concept','unit'],keep='last')
    for end,label in [(960,'12h'),(1680,'24h')]:
        post=labs[labs.event_minute.gt(240)&labs.event_minute.le(end)&labs.concept.str.startswith('troponin')]
        peak=post.groupby(['stay_id','concept','unit'],as_index=False).value.max().rename(columns={'value':'post_peak'})
        pairs=base_t.merge(peak,on=['stay_id','concept','unit'],how='inner')
        pairs['ratio']=pairs.post_peak/pairs.value
        result=pairs.groupby('stay_id').ratio.max()
        c['troponin_ratio_'+label]=result
        observed=c['troponin_ratio_'+label].notna()
        positive=c['troponin_ratio_'+label].ge(1.5)
        full=c.followup_end_offset_minutes.ge(end)
        c['troponin_rise_'+label]=positive.astype(float).where(observed&(positive|full))
        support.append(dict(dataset=d,endpoint='troponin_'+label,cohort=len(c),
            baseline_positive_assay_people=int(base_t.stay_id.nunique()),
            paired=int(observed.sum()),classifiable=int(c['troponin_rise_'+label].notna().sum()),
            events=int(c['troponin_rise_'+label].sum())))
    con.register('endpoint_frame',c.reset_index())
    con.execute(f'CREATE OR REPLACE TABLE {d}_context_endpoints AS SELECT * FROM endpoint_frame')
context_json('lab_assay_unit_audit.json',unit_rows)
context_json('endpoint_support.json',support)
print(pd.DataFrame(support).to_string(index=False))
context_status('raw source fingerprint verification')
for key,info in sources.items():
    p=Path(info['path']);digest=hashlib.sha256()
    with p.open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):digest.update(block)
    s=p.stat()
    assert s.st_size==info['bytes'] and s.st_mtime_ns==info['mtime_ns'], 'Source changed: '+key
    info['sha256']=digest.hexdigest()
context_json('input_manifest.json',sources)
context_status('SpO2-context features and endpoints extracted','complete')
con.close()
print('Feasibility outputs only: no new feature-outcome association or biological discovery claimed.')

eICU myocardial-injury and baseline lab extraction running
MIMIC myocardial-injury and baseline lab extraction running
dataset     endpoint  cohort  baseline_positive_assay_people  paired  classifiable  events
   eicu troponin_12h   12028                            3458    2539          2405     605
   eicu troponin_24h   12028                            3458    2648          2215     668
  mimic troponin_12h   17758                            4485    2858          2781     674
  mimic troponin_24h   17758                            4485    3033          2624     804
raw source fingerprint verification running
SpO2-context features and endpoints extracted complete
Feasibility outputs only: no new feature-outcome association or biological discovery claimed.


In [5]:
# Reconcile the historical recovery attempt to the exact production risk set.
# Raw scans can contain a superset; model tables must use the final cohort.
import json, importlib.util, sys
from pathlib import Path
from datetime import datetime, timezone
import duckdb
import pandas as pd
IN_COLAB=importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
DRIVE=Path('/content/drive/MyDrive') if IN_COLAB else Path.home()/'Library/CloudStorage/GoogleDrive-2arnavmana@gmail.com/My Drive'
PROJECT=DRIVE/'Projects'/'PhysioGraph'
OUT=PROJECT/'research'/'spo2_circulatory_context'
sys.path.insert(0,str(PROJECT/'src'))
from physiograph.analysis.spo2_protocol import _first_instability
(OUT/'run_status.json').write_text(json.dumps(dict(stage='production cohort and SpO2 boundary reconciliation',
    status='running',environment='google_colab' if IN_COLAB else 'local',
    updated_utc=datetime.now(timezone.utc).isoformat(),goal_complete=False),indent=2)+'\n')
con=duckdb.connect(str(DRIVE/'Data'/'PhysioGraph_Biological_Discovery_20260905'/'spo2_context.duckdb'))
con.execute('SET threads=2')
con.execute("SET memory_limit='2GB'")
# Recover the initial scan's omitted production warning filter exactly once.
# Fresh full-notebook extraction above already applies and records this rule.
quality_done=con.execute("SELECT count(*) FROM information_schema.tables WHERE table_name='mimic_context_quality_marker'").fetchone()[0]>0
if not quality_done:
    from physiograph.etl.mimic_extractor import _mimic_chartevent_item_map
    root=DRIVE/'Data'/'MIMIC'/'Full';raw_path=root/'chartevents.csv'
    input_info=json.loads((OUT/'input_manifest.json').read_text())['MIMIC/chartevents']
    st=raw_path.stat()
    assert st.st_size==input_info['bytes'] and st.st_mtime_ns==input_info['mtime_ns'], 'Raw input changed before quality correction'
    (OUT/'run_status.json').write_text(json.dumps(dict(stage='MIMIC production warning-filter correction',
        status='running',environment='google_colab' if IN_COLAB else 'local',
        updated_utc=datetime.now(timezone.utc).isoformat(),goal_complete=False),indent=2)+'\n')
    itemmap=_mimic_chartevent_item_map(root)
    mapping=pd.DataFrame([dict(itemid=i,concept=m) for i,m in itemmap.items()
                          if m in ['spo2','map','sbp','hr','resp_rate','fio2']])
    con.register('quality_vital_map',mapping)
    before_quality=con.execute('SELECT count(*) FROM mimic_vitals').fetchone()[0]
    con.execute("""CREATE OR REPLACE TABLE mimic_vitals AS
      SELECT c.stay_id,date_diff('second',c.admit_time,try_cast(v.charttime AS TIMESTAMP))/60.0 event_minute,
      m.concept,try_cast(v.valuenum AS DOUBLE) AS value,
      CASE WHEN try_cast(v.itemid AS BIGINT) IN (220181,220179) THEN 'noninvasive'
      WHEN m.concept IN ('map','sbp') THEN 'invasive' ELSE 'charted' END source_type
      FROM read_csv(?,header=true,all_varchar=true,sample_size=10000,strict_mode=true) v
      JOIN mimic_cohort c ON try_cast(v.stay_id AS BIGINT)=c.stay_id
      JOIN quality_vital_map m ON try_cast(v.itemid AS BIGINT)=m.itemid
      WHERE try_cast(v.charttime AS TIMESTAMP)>=c.admit_time
        AND try_cast(v.charttime AS TIMESTAMP)<c.admit_time+INTERVAL '4 hours'
        AND coalesce(try_cast(v.warning AS DOUBLE),0)<>1
        AND try_cast(v.valuenum AS DOUBLE) IS NOT NULL""",[str(raw_path)])
    con.execute("""CREATE OR REPLACE TABLE mimic_bins AS
      WITH dedup AS (
        SELECT stay_id,event_minute,concept,median(value) AS value FROM mimic_vitals
        WHERE (concept='spo2' AND value BETWEEN 50 AND 100)
          OR (concept='map' AND value BETWEEN 20 AND 200)
          OR (concept='hr' AND value BETWEEN 20 AND 250)
          OR (concept='sbp' AND value BETWEEN 30 AND 300)
          OR (concept='resp_rate' AND value BETWEEN 2 AND 80)
        GROUP BY stay_id,event_minute,concept)
      SELECT stay_id,floor(event_minute/15)::INTEGER bin,concept,median(value) AS value,
        median(event_minute) representative_minute,count(*) n_readings
        FROM dedup GROUP BY stay_id,bin,concept""")
    st=raw_path.stat()
    assert st.st_size==input_info['bytes'] and st.st_mtime_ns==input_info['mtime_ns'], 'Raw input changed during quality correction'
    con.execute('CREATE OR REPLACE TABLE mimic_context_quality_marker AS SELECT 1 AS version')
    (OUT/'mimic_quality_correction.json').write_text(json.dumps(dict(
        raw_rows_before=before_quality,raw_rows_after=con.execute('SELECT count(*) FROM mimic_vitals').fetchone()[0],
        correction='canonical cohort, [0,240) window, and warning !=1',
        production_filter='src/physiograph/etl/mimic_extractor.py:_stream_mimic_measurement_events',
        source_sha256=input_info['sha256'],cross_tabulations_previously_seen=True,
        no_effect_estimates_fitted=True),indent=2)+'\n')
reconciliation=[];support=[]
for d in ['eicu','mimic']:
    before=con.execute(f'SELECT count(*) FROM {d}_cohort').fetchone()[0]
    con.execute(f"""CREATE TABLE IF NOT EXISTS {d}_canonical_exclusions AS
    SELECT * FROM {d}_cohort WHERE FALSE""")
    con.execute(f"""INSERT INTO {d}_canonical_exclusions
    SELECT * FROM {d}_cohort WHERE death_offset_minutes BETWEEN 0 AND 240
      OR followup_end_offset_minutes<=240""")
    con.execute(f"DELETE FROM {d}_cohort WHERE stay_id IN (SELECT stay_id FROM {d}_canonical_exclusions)")
    for suffix in ['features','context_endpoints']:
        con.execute(f'DELETE FROM {d}_{suffix} WHERE stay_id NOT IN (SELECT stay_id FROM {d}_cohort)')
    # Production SpO2 dynamics use [0,240), unlike baseline labs which include 240.
    # The initial raw scan was an inclusive superset; remove its terminal bin.
    boundary_records=con.execute(f'SELECT count(*) FROM {d}_bins WHERE bin>=16').fetchone()[0]
    before_dynamics=con.execute(f'SELECT count(*) FROM {d}_features WHERE spo2_bins>=3 AND spo2_transitions>=2').fetchone()[0]
    con.execute(f'DELETE FROM {d}_bins WHERE bin>=16')
    con.execute(f"""CREATE OR REPLACE TABLE {d}_features AS
      WITH wide AS (
        SELECT b.stay_id,bin,max(value) FILTER(WHERE concept='spo2') spo2,
        max(value) FILTER(WHERE concept='map') map_value,
        max(value) FILTER(WHERE concept='hr') hr,
        max(representative_minute) FILTER(WHERE concept='spo2') spo2_time
        FROM {d}_bins b JOIN {d}_cohort c USING(stay_id) GROUP BY b.stay_id,bin), lagged AS (
        SELECT *,spo2-lag(spo2) OVER w ds,map_value-lag(map_value) OVER w dm,
        spo2_time-lag(spo2_time) OVER w gap FROM wide WHERE spo2 IS NOT NULL
        WINDOW w AS (PARTITION BY stay_id ORDER BY bin)), marked AS (
        SELECT *,coalesce(gap>0 AND gap<=30,FALSE) AS valid_transition,
        coalesce(gap>0 AND gap<=30 AND dm IS NOT NULL,FALSE) joint FROM lagged)
      SELECT stay_id,count(*) spo2_bins,count(*) FILTER(WHERE valid_transition) spo2_transitions,
      count(*) FILTER(WHERE joint) joint_transitions,
      count(*)>=3 AND count(*) FILTER(WHERE joint)>=2 joint_eligible,
      max(CASE WHEN valid_transition AND abs(ds)>=4 THEN 1 ELSE 0 END) spo2_jump_any,
      max(CASE WHEN joint AND abs(ds)>=4 THEN 1 ELSE 0 END) joint_spo2_jump_any,
      max(CASE WHEN joint AND dm<=-10 THEN 1 ELSE 0 END) map_fall_any,
      max(CASE WHEN joint AND abs(ds)>=4 AND dm<=-10 THEN 1 ELSE 0 END) synchronous_any,
      max(CASE WHEN joint AND ds<=-4 AND dm<=-10 THEN 1 ELSE 0 END) synchronous_drop,
      max(CASE WHEN joint AND ds>=4 AND dm<=-10 THEN 1 ELSE 0 END) synchronous_rise,
      count(*) FILTER(WHERE joint AND abs(ds)>=4) joint_spo2_jumps,
      count(*) FILTER(WHERE joint AND dm<=-10) map_falls,
      avg(spo2) spo2_mean,min(spo2) spo2_min,avg(map_value) map_mean,min(map_value) map_min,
      avg(hr) hr_mean,count(map_value) map_observed_bins FROM marked GROUP BY stay_id""")
    c=con.execute(f'SELECT * FROM {d}_cohort').df()
    f=con.execute(f'SELECT * FROM {d}_features').df()
    assert c.stay_id.is_unique and c.age.ge(18).all() and c.cohort_hf_flag.eq(1).all()
    assert (c.early_icu_flag.eq(1)|c.shock_icd_flag.eq(1)).all()
    assert not (c.death_offset_minutes.between(0,240)|c.followup_end_offset_minutes.le(240)).any()
    assert set(f.stay_id).issubset(set(c.stay_id))
    raw_spo2=con.execute(f"""SELECT v.stay_id,'{d}' dataset,concept,
        event_minute offset_minutes,value value_numeric FROM {d}_vitals v
        JOIN {d}_cohort c USING(stay_id) WHERE concept='spo2'""").df()
    production_exposed=set(_first_instability(raw_spo2).stay_id)
    rebuilt_exposed=set(f.loc[f.spo2_bins.ge(3)&f.spo2_transitions.ge(2)&f.spo2_jump_any.eq(1),'stay_id'])
    assert production_exposed==rebuilt_exposed, 'Production SpO2 exposure mismatch: '+d
    eligible=f[f.joint_eligible]
    reconciliation.append(dict(dataset=d,before=before,canonical=len(c),removed_this_execution=before-len(c),
        terminal_bins_removed=boundary_records,dynamics_before_boundary_fix=before_dynamics,
        dynamics_after_boundary_fix=int(((f.spo2_bins>=3)&(f.spo2_transitions>=2)).sum()),
        production_exposure_patient_sets_match=True,production_exposed=len(production_exposed),
        cumulative_excluded=con.execute(f'SELECT count(*) FROM {d}_canonical_exclusions').fetchone()[0],
        historical_reference={'eicu':12028,'mimic':17758}[d],
        matches_historical_count=len(c)=={'eicu':12028,'mimic':17758}[d]))
    support.append(dict(dataset=d,cohort=len(c),spo2_observed=len(f),
        spo2_dynamics=int(((f.spo2_bins>=3)&(f.spo2_transitions>=2)).sum()),
        jointly_observed=len(eligible),synchronous=int(eligible.synchronous_any.sum()),
        simultaneous_drop=int(eligible.synchronous_drop.sum()),simultaneous_rise=int(eligible.synchronous_rise.sum())))
(OUT/'canonical_cohort_reconciliation.json').write_text(json.dumps(reconciliation,indent=2)+'\n')
(OUT/'joint_feature_support_canonical.json').write_text(json.dumps(support,indent=2)+'\n')
print(pd.DataFrame(reconciliation).to_string(index=False))
print(pd.DataFrame(support).to_string(index=False))
con.close()

dataset  before  canonical  removed_this_execution  terminal_bins_removed  dynamics_before_boundary_fix  dynamics_after_boundary_fix  production_exposure_patient_sets_match  production_exposed  cumulative_excluded  historical_reference  matches_historical_count
   eicu   12028      12028                       0                      0                         11452                        11452                                    True                4309                    0                 12028                      True
  mimic   17758      17758                       0                      0                          4711                         4711                                    True                1398                   18                 17758                      True
dataset  cohort  spo2_observed  spo2_dynamics  jointly_observed  synchronous  simultaneous_drop  simultaneous_rise
   eicu   12028          11572          11452              1022           76                 36    

In [6]:
# Pressure-source sensitivity and timing support, without outcome associations.
import json, importlib.util
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
IN_COLAB=importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
DRIVE=Path('/content/drive/MyDrive') if IN_COLAB else Path.home()/'Library/CloudStorage/GoogleDrive-2arnavmana@gmail.com/My Drive'
PROJECT=DRIVE/'Projects'/'PhysioGraph'
OUT=PROJECT/'research'/'spo2_circulatory_context'
con=duckdb.connect(str(DRIVE/'Data'/'PhysioGraph_Biological_Discovery_20260905'/'spo2_context.duckdb'))
con.execute('SET threads=2')
summaries=[];provenance=[]
for d in ['eicu','mimic']:
    provenance.extend(con.execute(f"""SELECT '{d}' dataset,source_type,count(*) records,
        count(DISTINCT v.stay_id) stays FROM {d}_vitals v JOIN {d}_cohort c USING(stay_id)
        WHERE concept='map' AND value BETWEEN 20 AND 200 GROUP BY source_type""").df().to_dict('records'))
    for variant in (['pooled','arterial'] if d=='mimic' else ['pooled']):
        if variant=='pooled':
            b=con.execute(f'SELECT b.* FROM {d}_bins b JOIN {d}_cohort c USING(stay_id)').df()
        else:
            b=con.execute(f"""WITH dedup AS (
              SELECT v.stay_id,event_minute,concept,median(value) AS value
              FROM {d}_vitals v JOIN {d}_cohort c USING(stay_id)
              WHERE event_minute<240 AND ((concept='spo2' AND value BETWEEN 50 AND 100)
                OR (concept='map' AND source_type='invasive' AND value BETWEEN 20 AND 200)
                OR (concept='hr' AND value BETWEEN 20 AND 250))
              GROUP BY v.stay_id,event_minute,concept)
              SELECT stay_id,floor(event_minute/15)::INTEGER bin,concept,
              median(value) AS value,median(event_minute) representative_minute,
              count(*) n_readings FROM dedup GROUP BY stay_id,bin,concept""").df()
        wide=b.pivot(index=['stay_id','bin'],columns='concept',values='value').reset_index()
        for concept in ['spo2','map']:
            clock=b[b.concept.eq(concept)][['stay_id','bin','representative_minute','n_readings']]
            clock=clock.rename(columns={'representative_minute':concept+'_time','n_readings':concept+'_readings'})
            wide=wide.merge(clock,on=['stay_id','bin'],how='left',validate='one_to_one')
        s=wide[wide.spo2.notna()].sort_values(['stay_id','bin']).copy()
        g=s.groupby('stay_id',sort=False)
        gap=g.spo2_time.diff();ds=g.spo2.diff();dm=g['map'].diff()
        s['valid']=gap.gt(0)&gap.le(30);s['joint']=s.valid&dm.notna()
        s['jump']=ds.abs().ge(4)&s.joint;s['fall']=dm.le(-10)&s.joint
        s['sync']=s.jump&s.fall;s['sync_drop']=s.sync&ds.lt(0);s['sync_rise']=s.sync&ds.gt(0)
        s['low_map']=s['map'].lt(65)
        f=s.groupby('stay_id',sort=False).agg(
            spo2_bins=('spo2','count'),spo2_transitions=('valid','sum'),joint_transitions=('joint','sum'),
            joint_spo2_jump_any=('jump','max'),map_fall_any=('fall','max'),
            synchronous_any=('sync','max'),synchronous_drop=('sync_drop','max'),synchronous_rise=('sync_rise','max'),
            joint_spo2_jumps=('jump','sum'),map_falls=('fall','sum'),
            spo2_mean=('spo2','mean'),spo2_min=('spo2','min'),map_mean=('map','mean'),map_min=('map','min'),
            hr_mean=('hr','mean'),spo2_readings=('spo2_readings','sum'),map_readings=('map_readings','sum'),
            low_map_bins=('low_map','sum'),map_observed_bins=('map','count')).reset_index()
        f['joint_eligible']=f.spo2_bins.ge(3)&f.joint_transitions.ge(2)
        for col in ['joint_spo2_jump_any','map_fall_any','synchronous_any','synchronous_drop','synchronous_rise']:
            f[col]=f[col].astype(int)
        t=pd.DataFrame(dict(stay_id=s.stay_id,bin=s.bin,event_minute=s.spo2_time,
            spo2_change=ds,map_change=dm,joint=s.joint,spo2_jump=s.jump,map_fall=s.fall,
            map_spo2_time_difference=s.map_time-s.spo2_time))
        t=t[t.joint].copy()
        # Independent SQL reconstruction on actual data checks the transition logic.
        con.register('check_bins',b)
        check=con.execute("""WITH wide AS (
          SELECT stay_id,bin,max(value) FILTER(WHERE concept='spo2') spo2,
          max(value) FILTER(WHERE concept='map') map_value,
          max(representative_minute) FILTER(WHERE concept='spo2') spo2_time
          FROM check_bins GROUP BY stay_id,bin), lagged AS (
          SELECT *,spo2-lag(spo2) OVER w ds,map_value-lag(map_value) OVER w dm,
          spo2_time-lag(spo2_time) OVER w gap FROM wide WHERE spo2 IS NOT NULL
          WINDOW w AS (PARTITION BY stay_id ORDER BY bin)), marked AS (
          SELECT *,coalesce(gap>0 AND gap<=30 AND dm IS NOT NULL,FALSE) joint
          FROM lagged)
          SELECT stay_id,count(*) spo2_bins,count(*) FILTER(WHERE joint) joint_transitions,
          count(*) FILTER(WHERE joint AND abs(ds)>=4) joint_spo2_jumps,
          count(*) FILTER(WHERE joint AND dm<=-10) map_falls,
          max(CASE WHEN joint AND abs(ds)>=4 AND dm<=-10 THEN 1 ELSE 0 END) synchronous_any
          FROM marked GROUP BY stay_id""").df().set_index('stay_id')
        actual=f.set_index('stay_id').reindex(check.index)
        for col in check.columns:
            assert actual[col].eq(check[col]).all(), 'Independent SQL feature mismatch: '+d+'/'+variant+'/'+col
        con.register('source_features',f);con.register('source_transitions',t)
        con.execute(f'CREATE OR REPLACE TABLE {d}_{variant}_features AS SELECT * FROM source_features')
        con.execute(f'CREATE OR REPLACE TABLE {d}_{variant}_transitions AS SELECT * FROM source_transitions')
        if variant=='pooled':
            old=con.execute(f'SELECT * FROM {d}_features WHERE spo2_bins>0').df().set_index('stay_id')
            new=f.set_index('stay_id').reindex(old.index)
            for col in ['joint_eligible','spo2_bins','spo2_transitions','joint_transitions',
                        'joint_spo2_jump_any','map_fall_any','synchronous_any','synchronous_drop','synchronous_rise']:
                assert new[col].eq(old[col]).all(), 'Feature definition drift: '+d+'/'+col
        a=f[f.joint_eligible];both=a.joint_spo2_jump_any.eq(1)&a.map_fall_any.eq(1)
        summaries.append(dict(dataset=d,variant=variant,jointly_observed=len(a),
            both_component_events=int(both.sum()),synchronous=int(a.synchronous_any.sum()),
            asynchronous_both=int((both&a.synchronous_any.eq(0)).sum()),
            joint_transitions=len(t),
            median_absolute_within_bin_time_difference=float(t.map_spo2_time_difference.abs().median())))
(OUT/'pressure_provenance.json').write_text(json.dumps(provenance,indent=2)+'\n')
(OUT/'pressure_source_support.json').write_text(json.dumps(summaries,indent=2,allow_nan=False)+'\n')
print(pd.DataFrame(summaries).to_string(index=False))
con.close()

dataset  variant  jointly_observed  both_component_events  synchronous  asynchronous_both  joint_transitions  median_absolute_within_bin_time_difference
   eicu   pooled              1022                    185           76                109              11766                                         0.0
  mimic   pooled              2846                    477          223                254              17645                                         0.0
  mimic arterial              1061                    105           51                 54               5612                                         0.0


# Circulatory-context analysis specification

Written before new exposure–outcome cross-tabulations or model fits on
2026-09-05 UTC. The raw scans are running. Feature and total endpoint
availability may be inspected; no effect estimates have been inspected.
This supplements SPO2_CIRCULATORY_CONTEXT_PLAN.md without changing its
thresholds, cohort, or biological question.

## Estimand and support gate

The primary timing comparison is **synchronous versus asynchronous events
among encounters having both a qualifying SpO2 change and a qualifying MAP
fall during the first four hours**. It is not synchrony versus every other
patient. The latter would mostly compare different component abnormalities
and would require impossible counterfactual synchrony in patients without
either component. Report the four component states descriptively.

Primary outcomes are hospital mortality and the existing troponin-rise
endpoint during 12 hours after the four-hour landmark (ICU minutes 240–960).
The 24-hour endpoint is a sensitivity analysis; a negative or sparse 12-hour
result cannot be rescued by selecting 24 hours. Troponin requires a positive
baseline value and a subsequent value of the same assay and unit, with rise
ratio at least 1.5. Without a detected rise, follow-up must extend through
the horizon. Missing or incompletely observed outcomes are not negatives.

Evaluate the previously frozen event gate **within the primary timing
comparison**, separately in each database: at least 100 events overall and
at least 20 events and 20 non-events in each timing group. Otherwise produce
counts only, explicitly fail the primary fitting gate, and do not lower it.
Also report marginal event support in all jointly observed patients to make
the source of attrition clear. No group with fewer than five patients is
used to support a biological claim.

## Model sequence if support is adequate

First quantify a conditional association with a binomial logit model.
The base model contains age, sex, mean heart rate, mean and minimum SpO2,
mean and minimum MAP, counts of SpO2 jumps and MAP falls, count of jointly
observed transitions, and log(1 + raw SpO2 reading count). The extended
model adds the binary synchrony indicator. Numeric predictors are centered
and scaled within each source; sex is categorical. Missing heart rate uses
the source median and an explicit missing indicator. Other required
physiological values must be observed. No predictor selection by p-value.

Use person-clustered covariance within each source, and an eICU sensitivity
with hospital-clustered covariance when at least 30 hospitals contribute.
Check model convergence, design rank, complete separation, and adequate
events for the number of fitted coefficients (at least 10 events and 10
non-events per coefficient). Failure means no model claim. Report the
conditional odds ratio and 95% confidence interval, explicitly as an odds
ratio. Do not relabel it as a risk ratio or causal effect. Correct the two
primary endpoint tests by Holm within each database. Replication requires
the same direction and multiplicity-adjusted evidence in both sources.

This first model is a **screen**, not the final biological test. Advancement
requires baseline respiratory support, oxygen therapy and vasoactive
treatment controls from original sources, observed baseline lactate and
creatinine, pressure-source sensitivity, ordering/measurement selection
analysis for troponin, and evaluation of predictive improvement. No
unadjusted or screen-only association is promoted as a discovery.

## Fixed sensitivity checks

- Repeat MIMIC features using arterial MAP only. eICU's supplied periodic
  pressure source is arterial; do not infer unmeasured cuff pressure.
- Analyze downward and upward SpO2 changes separately without choosing a
  preferred direction after seeing results.
- Evaluate the predeclared all-SpO2-bins-at-least-90% and measured baseline
  lactate-below-2 mmol/L subgroups only if they pass the same support gate.
  Retrospective specimen-time and actual result-availability definitions
  must be distinguished. eICU revision time is not proven first availability.
- Compare within-bin measurement time differences across pressure sources.
  Same-bin synchrony is coarse charted synchrony, not beat-to-beat coupling.

Low perfusion can alter pulse-oximeter measurement. Neither synchrony nor
troponin elevation establishes myocardial hypoxia, reduced cardiac output,
or a treatable mechanism. Reject significance that depends on a changed
threshold, an unsupported subgroup, a missing-data convention, or a single
database. The paradigm-shifting discovery goal remains unfulfilled unless
substantially stronger evidence is actually obtained.

## Prior-art check

The joint prognostic relevance of low saturation and low blood pressure in
acute HF was already reported in a [2009 clinical cohort](https://pubmed.ncbi.nlm.nih.gov/19372679/).
Combining those two absolute levels is therefore not a new biological finding.
A [2021 registered protocol](https://cdn.clinicaltrials.gov/large-docs/01/NCT05188001/Prot_SAP_000.pdf)
also proposed vital-sign time series for predicting troponin and myocardial
injury after noncardiac surgery; this is adjacent prior art, not an HF result
or proof that its planned analyses succeeded. The narrower timing contrast
in this extension remains a hypothesis. A search finding no exact match
does not establish novelty.

In [7]:
ANALYSIS_LOCK_SHA256='8fc4435812e6456585732f940a4a97e8c83fe86551226475abea33e142582dba'

In [8]:
# Apply the frozen support gate before estimating any association.
import json, importlib.util
from pathlib import Path
from datetime import datetime, timezone
import duckdb
import pandas as pd
IN_COLAB=importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
DRIVE=Path('/content/drive/MyDrive') if IN_COLAB else Path.home()/'Library/CloudStorage/GoogleDrive-2arnavmana@gmail.com/My Drive'
PROJECT=DRIVE/'Projects'/'PhysioGraph'
OUT=PROJECT/'research'/'spo2_circulatory_context'
lock_path=OUT/'analysis_lock.json'
if lock_path.exists():
    assert json.loads(lock_path.read_text())['sha256']==ANALYSIS_LOCK_SHA256, 'Analysis specification changed after locking'
else:
    lock_path.write_text(json.dumps(dict(sha256=ANALYSIS_LOCK_SHA256,
        recorded_before_cross_tabulations=True,utc=datetime.now(timezone.utc).isoformat()),indent=2)+'\n')
con=duckdb.connect(str(DRIVE/'Data'/'PhysioGraph_Biological_Discovery_20260905'/'spo2_context.duckdb'))
gates=[];states=[];endpoint_support=[]
for d in ['eicu','mimic']:
    endpoints=con.execute(f'SELECT * FROM {d}_context_endpoints').df()
    for endpoint in ['hospital_mortality','troponin_rise_12h','troponin_rise_24h']:
        endpoint_support.append(dict(dataset=d,endpoint=endpoint,cohort=len(endpoints),
            observed=int(endpoints[endpoint].notna().sum()),events=int(endpoints[endpoint].sum())))
    for variant in (['pooled','arterial'] if d=='mimic' else ['pooled']):
        f=con.execute(f'SELECT * FROM {d}_{variant}_features WHERE joint_eligible').df()
        a=f.merge(endpoints,on='stay_id',validate='one_to_one')
        a['component_state']=a.joint_spo2_jump_any.astype(int)+2*a.map_fall_any.astype(int)
        for endpoint in ['hospital_mortality','troponin_rise_12h','troponin_rise_24h']:
            observed=a[a[endpoint].notna()]
            for state,g in observed.groupby('component_state'):
                states.append(dict(dataset=d,variant=variant,endpoint=endpoint,
                    component_state={0:'neither',1:'spo2_only',2:'map_only',3:'both'}[state],
                    n=len(g),events=int(g[endpoint].sum())))
            for subgroup in ['all','all_spo2_ge90','measured_lactate_lt2']:
                b=observed[observed.component_state.eq(3)]
                if subgroup=='all_spo2_ge90':b=b[b.spo2_min.ge(90)]
                elif subgroup=='measured_lactate_lt2':b=b[b.baseline_lactate.lt(2)]
                sync=b[b.synchronous_any.eq(1)];async_group=b[b.synchronous_any.eq(0)]
                es=int(sync[endpoint].sum());ea=int(async_group[endpoint].sum())
                nes=len(sync)-es;nea=len(async_group)-ea
                sufficient=es+ea>=100 and min(es,ea,nes,nea)>=20
                gates.append(dict(dataset=d,variant=variant,endpoint=endpoint,subgroup=subgroup,
                    jointly_observed_with_outcome=len(observed),primary_comparison_n=len(b),
                    events=es+ea,synchronous_n=len(sync),synchronous_events=es,
                    asynchronous_n=len(async_group),asynchronous_events=ea,
                    passes_count_gate=bool(sufficient),
                    decision='eligible for model support checks' if sufficient else 'counts only; frozen event gate failed'))
for name,obj in [('timing_contrast_support.json',gates),('component_state_counts.json',states),
                 ('endpoint_support_canonical.json',endpoint_support)]:
    (OUT/name).write_text(json.dumps(obj,indent=2,allow_nan=False)+'\n')
print(pd.DataFrame(gates).query("subgroup == 'all'").to_string(index=False))
con.close()
print('Support gate evaluated locally; no model fit or biological discovery asserted by this cell.')

dataset  variant           endpoint subgroup  jointly_observed_with_outcome  primary_comparison_n  events  synchronous_n  synchronous_events  asynchronous_n  asynchronous_events  passes_count_gate                              decision
   eicu   pooled hospital_mortality      all                           1013                   184      57             76                  24             108                   33              False counts only; frozen event gate failed
   eicu   pooled  troponin_rise_12h      all                            190                    43      29             19                  14              24                   15              False counts only; frozen event gate failed
   eicu   pooled  troponin_rise_24h      all                            191                    47      30             20                  14              27                   16              False counts only; frozen event gate failed
  mimic   pooled hospital_mortality      all                

In [9]:
# Conditional association screen; it runs only when the frozen gates pass.
import json, importlib.util, subprocess, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
OUT=PROJECT/'research'/'spo2_circulatory_context'
gates=json.loads((OUT/'timing_contrast_support.json').read_text())
primary=[g for g in gates if g['variant']=='pooled' and g['subgroup']=='all'
         and g['endpoint'] in ['hospital_mortality','troponin_rise_12h']]
results=[]
if any(g['passes_count_gate'] for g in primary):
    if importlib.util.find_spec('statsmodels') is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','statsmodels==0.14.4','scipy==1.15.2'])
    import statsmodels.api as sm
    from statsmodels.tools.sm_exceptions import PerfectSeparationWarning
    from statsmodels.stats.multitest import multipletests
    con=duckdb.connect(str(DRIVE/'Data'/'PhysioGraph_Biological_Discovery_20260905'/'spo2_context.duckdb'))
for gate in primary:
    result=dict(dataset=gate['dataset'],endpoint=gate['endpoint'],stage='conditional association screen',
                causal=False,treatment_adjusted=False,biological_discovery=False)
    if not gate['passes_count_gate']:
        result.update(status='not fitted',reason='frozen count gate failed');results.append(result);continue
    d=gate['dataset'];endpoint=gate['endpoint']
    f=con.execute(f'SELECT * FROM {d}_pooled_features WHERE joint_eligible AND joint_spo2_jump_any=1 AND map_fall_any=1').df()
    e=con.execute(f'SELECT * FROM {d}_context_endpoints').df()
    a=f.merge(e,on='stay_id',validate='one_to_one').dropna(subset=[endpoint]).copy()
    numeric=['age','hr_mean','spo2_mean','spo2_min','map_mean','map_min',
             'joint_spo2_jumps','map_falls','joint_transitions']
    a['hr_missing']=a.hr_mean.isna().astype(float)
    a['hr_mean']=a.hr_mean.fillna(a.hr_mean.median())
    a=a.dropna(subset=numeric+['spo2_readings','person_id'])
    x=a[numeric].astype(float).copy()
    x['log_spo2_readings']=np.log1p(a.spo2_readings)
    for col in x:
        sd=x[col].std(ddof=0)
        x[col]=(x[col]-x[col].mean())/sd if sd>0 else 0.0
    sex=a.is_male.map({0.0:'female',1.0:'male'}).fillna('unknown')
    x=pd.concat([x,pd.get_dummies(sex,prefix='sex',drop_first=True,dtype=float)],axis=1)
    x['hr_missing']=a.hr_missing
    x=x.loc[:,x.nunique()>1]
    x=sm.add_constant(x,has_constant='add');x['synchronous_any']=a.synchronous_any.astype(float)
    y=a[endpoint].astype(float)
    grouped=a.groupby('synchronous_any')[endpoint].agg(['count','sum'])
    event_gate=len(grouped)==2 and y.sum()>=100 and grouped['sum'].min()>=20 and (grouped['count']-grouped['sum']).min()>=20
    result.update(n=len(a),events=int(y.sum()),coefficients=x.shape[1],person_clusters=int(a.person_id.nunique()))
    reason=None
    if not event_gate:reason='count gate failed after required-covariate exclusions'
    elif min(y.sum(),len(y)-y.sum())<10*x.shape[1]:reason='fewer than 10 events or non-events per coefficient'
    elif a.person_id.nunique()<30:reason='fewer than 30 independent person clusters'
    elif np.linalg.matrix_rank(x.to_numpy())<x.shape[1]:reason='rank deficient design'
    if reason:
        result.update(status='not fitted',reason=reason);results.append(result);continue
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('error',PerfectSeparationWarning)
            base=sm.GLM(y,x.drop(columns='synchronous_any'),family=sm.families.Binomial()).fit(maxiter=100)
            model=sm.GLM(y,x,family=sm.families.Binomial()).fit(maxiter=100,
                cov_type='cluster',cov_kwds={'groups':a.person_id})
        if not base.converged or not model.converged:raise ValueError('Model did not converge')
        beta=float(model.params['synchronous_any']);se=float(model.bse['synchronous_any'])
        if not np.isfinite([beta,se]).all() or se<=0:raise ValueError('Invalid clustered uncertainty')
        if max(abs(beta-1.96*se),abs(beta+1.96*se))>700:raise ValueError('Unstable, unbounded effect interval')
        result.update(status='fitted',conditional_odds_ratio=float(np.exp(beta)),
            ci95=[float(np.exp(beta-1.96*se)),float(np.exp(beta+1.96*se))],
            p_value=float(model.pvalues['synchronous_any']),
            base_deviance=float(base.deviance),extended_deviance=float(model.deviance))
        if d=='eicu' and a.hospital_id.nunique()>=30:
            hospital=sm.GLM(y,x,family=sm.families.Binomial()).fit(maxiter=100,
                cov_type='cluster',cov_kwds={'groups':a.hospital_id})
            hs=float(hospital.bse['synchronous_any'])
            result['hospital_cluster_ci95']=[float(np.exp(beta-1.96*hs)),float(np.exp(beta+1.96*hs))]
    except (ValueError,np.linalg.LinAlgError,PerfectSeparationWarning) as exc:
        result.update(status='fit failed',reason=str(exc))
    results.append(result)
if any(g['passes_count_gate'] for g in primary):
    con.close()
    for d in ['eicu','mimic']:
        rows=[r for r in results if r['dataset']==d]
        adjusted=multipletests([r.get('p_value',1.0) if r['status']=='fitted' else 1.0 for r in rows],method='holm')[1]
        for r,p in zip(rows,adjusted):
            if r['status']=='fitted':r['holm_p_two_primary_endpoints']=float(p)
(OUT/'conditional_screen_results.json').write_text(json.dumps(results,indent=2,allow_nan=False)+'\n')
replicated=[]
for endpoint in ['hospital_mortality','troponin_rise_12h']:
    pair=[r for r in results if r['endpoint']==endpoint]
    if len(pair)==2 and all(r['status']=='fitted' and r['conditional_odds_ratio']>1
                          and r.get('holm_p_two_primary_endpoints',1)<.05 for r in pair):
        replicated.append(endpoint)
from datetime import datetime,timezone
import hashlib
finished=datetime.now(timezone.utc).isoformat()
manifest=dict(environment='google_colab' if IN_COLAB else 'local',finished_utc=finished,
    analysis_lock_sha256=ANALYSIS_LOCK_SHA256,synthetic_patients=0,
    raw_input_manifest_sha256=hashlib.sha256((OUT/'input_manifest.json').read_bytes()).hexdigest(),
    implementation_sha256=hashlib.sha256((PROJECT/'scripts/build_biological_notebook.py').read_bytes()).hexdigest(),
    production_spo2_helper_sha256=hashlib.sha256((PROJECT/'src/physiograph/analysis/spo2_protocol.py').read_bytes()).hexdigest(),
    software=dict(python=sys.version,numpy=np.__version__,pandas=pd.__version__,duckdb=duckdb.__version__),
    cross_database_positive_conditional_screens=replicated,
    paradigm_shifting_result=False,clinical_benefit_established=False)
(OUT/'analysis_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
(OUT/'run_status.json').write_text(json.dumps(dict(stage='SpO2 circulatory-context candidate analyzed',
    status='complete',environment=manifest['environment'],updated_utc=finished,goal_complete=False,
    cross_database_positive_conditional_screens=replicated),indent=2)+'\n')
print(pd.DataFrame(results).to_string(index=False))
print('This is a conditional association screen. The biological discovery goal is not complete.')

dataset           endpoint                          stage  causal  treatment_adjusted  biological_discovery     status                   reason
   eicu hospital_mortality conditional association screen   False               False                 False not fitted frozen count gate failed
   eicu  troponin_rise_12h conditional association screen   False               False                 False not fitted frozen count gate failed
  mimic hospital_mortality conditional association screen   False               False                 False not fitted frozen count gate failed
  mimic  troponin_rise_12h conditional association screen   False               False                 False not fitted frozen count gate failed
This is a conditional association screen. The biological discovery goal is not complete.


# Biological direction of the SpO2–troponin association

Prospective specification for a new mechanistic falsification analysis,
2026-09-05 UTC. Patient data are restricted to MIMIC and eICU and the
original PhysioGraph HF cohorts. This is not a replacement threshold for
the failed MAP-synchrony candidate.

The current review's strongest two-source secondary signal is troponin
rise, but its advanced estimates disagree. The biological ordering of
myocardial injury and the first SpO2 episode remains unresolved. Delayed
troponin detection can follow injury that began before the respiratory
signal. Troponin kinetics do not establish infarction mechanism:
[primary clinical kinetics study](https://pmc.ncbi.nlm.nih.gov/articles/8360674/).

## First gate: can the biological ordering be observed?

Retain the production first absolute SpO2 change of at least four percentage
points, in 15-minute medians over [0,240) ICU minutes, with at least three
bins and two transitions separated by no more than 30 minutes. Use the
warning-filtered MIMIC replay and the current eICU periodic data. Do not
condition this question on having MAP measurements.

Collect troponin T and I measurements from the same hospital encounter from
24 hours before ICU admission through 28 hours afterward. Do not include
samples before the hospital encounter began. Preserve assay, unit, specimen
time and result/revision time; the latter are not interchangeable. Reuse
verified extracted post-admission samples and obtain only the missing
pre-admission-to-ICU segment from the original laboratory sources.

At the first SpO2 episode, examine the preceding 12 hours and following
12 hours. A serial pre-episode trajectory requires at least two positive
values of the same assay and unit separated by at least 60 minutes.
Choose the assay/unit with the earliest pre-episode measurement, breaking
ties by troponin T before I and then lexical unit. This selection uses
pre-episode data only. Define pre-episode rise as last/first >=1.5, preserving
the project's existing relative-rise threshold. Record the continuous
ratio as well. A post-episode trajectory requires a specimen at least
120 minutes after the episode and no later than 12 hours after it. The
post-episode ratio uses its maximum and the last pre-episode concentration.

First report only the numbers with an episode, pre-episode assays, serial
pre-episode measurements, and delayed post-episode measurements. Do not
inspect trajectory direction or join mortality in this first gate. At
least 100 exposed encounters with complete pre/post trajectories in EACH
database are required before advancing to direction estimates. If this
gate fails, retain the counts and stop this mechanistic candidate; do not
relax the separation, extend the window, or call one pre-episode sample a
stable trajectory.

If the gate passes, freeze the final directional contrast, uncertainty,
clinical controls, and mortality analysis before opening those outcomes.
Within-person temporal ordering alone cannot establish a treatment effect
or a paradigm-shifting biological discovery. An absent observed pre-episode
rise cannot rule out injury because biomarker appearance is delayed.

## Execution and reporting

Keep the runnable workflow in the existing single primary notebook.
Private patient-level tables remain under Data outside the repository.
Record source fingerprints, exact time/assay rules, exclusions and all failed
support gates. Label local runs accurately. No mortality improvement or
novel biological mechanism is presumed.


## Encounter-boundary amendment, before trajectory directions

The first feasibility pass used MIMIC `admittime` as encounter start and
found 17 complete trajectories. A source check then found that 987 of the
1,398 exposed encounters had an earlier `edregtime`, with a median 190
minutes from ED registration to inpatient admission. The [official MIMIC
documentation](https://mimic.mit.edu/docs/iv/modules/hosp/admissions.html)
distinguishes ED registration from inpatient admission.

For the revised feasibility pass, define MIMIC encounter start as the earlier
of `edregtime` and `admittime` from the same `hadm_id` row. Continue to require
an explicit matching laboratory `hadm_id`; do not assign unlinked samples by
proximity or by patient identity alone. Retain the original 24-hour pre-ICU
extraction bound, 12-hour pre/post-episode windows, 60-minute pre-sample
separation, 120-minute post-sample delay, assay-selection rule and 100-complete-
trajectory gate in both databases. eICU's hospital boundary is unchanged.

This amendment follows inspection of feasibility counts and source metadata,
but precedes inspection of trajectory directions, ratios or mortality.
Preserve the original specification, lock and results. This is a corrected
feasibility assessment, not a claim that the ED rule was prospectively frozen
before the first support pass or that a failed significance test was rescued.

In [10]:
DIRECTION_PROTOCOL_SHA256='cb137c6444593ddfa99eff90fd85c0216f92750c33f8f9c0d25290774f140490'
DIRECTION_BOUNDARY_SHA256='e9105d2d2973387c0181882f02e006d284d297a70efa73f4d34dc5b6b3e1fc2b'

In [11]:
# Biological-ordering feasibility; no trajectory ratios or mortality associations.
import sys,json,importlib.util
from pathlib import Path
from datetime import datetime,timezone
import duckdb
import pandas as pd
IN_COLAB=importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
DRIVE=Path('/content/drive/MyDrive') if IN_COLAB else Path.home()/'Library/CloudStorage/GoogleDrive-2arnavmana@gmail.com/My Drive'
PROJECT=DRIVE/'Projects'/'PhysioGraph'
sys.path.insert(0,str(PROJECT/'src'))
from physiograph.analysis.spo2_protocol import _first_instability
from physiograph.etl.mimic_extractor import _mimic_lab_item_map
from physiograph.constants import EICU_LAB_NAME_MAP
PRIVATE=DRIVE/'Data'/'PhysioGraph_Biological_Discovery_20260905'
OUT=PROJECT/'research'/'spo2_troponin_direction';OUT.mkdir(exist_ok=True)
PRIOR=PROJECT/'research'/'spo2_circulatory_context'
def direction_json(name,obj):
    (OUT/name).write_text(json.dumps(obj,indent=2,allow_nan=False,default=str)+'\n')
def direction_status(stage,status='running'):
    direction_json('run_status.json',dict(stage=stage,status=status,
        environment='google_colab' if IN_COLAB else 'local',
        updated_utc=datetime.now(timezone.utc).isoformat(),goal_complete=False))
    print(stage,status,flush=True)
lock=OUT/'protocol_lock.json'
if lock.exists():assert json.loads(lock.read_text())['sha256']==DIRECTION_PROTOCOL_SHA256
else:direction_json('protocol_lock.json',dict(sha256=DIRECTION_PROTOCOL_SHA256,
    locked_before_support_extraction=True,utc=datetime.now(timezone.utc).isoformat()))
amendment=OUT/'boundary_amendment.json'
if amendment.exists():assert json.loads(amendment.read_text())['sha256']==DIRECTION_BOUNDARY_SHA256
else:direction_json('boundary_amendment.json',dict(sha256=DIRECTION_BOUNDARY_SHA256,
    initial_support_counts_already_seen=True,trajectory_directions_not_seen=True,
    utc=datetime.now(timezone.utc).isoformat()))
con=duckdb.connect(str(PRIVATE/'troponin_direction.duckdb'))
con.execute('SET threads=2');con.execute("SET memory_limit='2GB'")
def qs(value):return "'"+str(value).replace("'","''")+"'"
con.execute(f"ATTACH {qs(PRIVATE/'spo2_context.duckdb')} AS prior (READ_ONLY)")
assert con.execute("SELECT count(*) FROM information_schema.tables WHERE table_catalog='prior' AND table_name='mimic_context_quality_marker'").fetchone()[0]>0
manifest=json.loads((PRIOR/'input_manifest.json').read_text())
used={}
def verified_source(dataset,name):
    key=dataset+'/'+name;info=manifest[key];p=Path(info['path']);st=p.stat()
    assert st.st_size==info['bytes'] and st.st_mtime_ns==info['mtime_ns'], 'Source changed: '+key
    used[key]=info
    return p
direction_status('episode anchors and hospital boundaries')
coverage=[]
for d,source in [('eicu','eICU'),('mimic','MIMIC')]:
    fields='stay_id,followup_end_offset_minutes'+(',admit_time,hadm_id' if d=='mimic' else '')
    c=con.execute(f'SELECT {fields} FROM prior.{d}_cohort').df()
    raw=con.execute(f"""SELECT v.stay_id,'{d}' dataset,concept,
        event_minute offset_minutes,value value_numeric FROM prior.{d}_vitals v
        JOIN prior.{d}_cohort c USING(stay_id) WHERE concept='spo2'""").df()
    onset=_first_instability(raw).rename(columns={'instability_onset_minutes':'episode_minute'})
    a=c.merge(onset[['stay_id','episode_minute']],on='stay_id',validate='one_to_one')
    if d=='eicu':
        patient=pd.read_csv(verified_source(source,'patient'),usecols=['patientunitstayid','hospitaladmitoffset'])
        starts=patient.set_index('patientunitstayid').hospitaladmitoffset
        a['hospital_start_minute']=pd.to_numeric(a.stay_id.map(starts),errors='coerce')
    else:
        admission=pd.read_csv(verified_source(source,'admissions'),usecols=['hadm_id','admittime','edregtime']).set_index('hadm_id')
        clocks=admission[['admittime','edregtime']].apply(pd.to_datetime,errors='coerce')
        starts=clocks.min(axis=1)
        a['hospital_start_minute']=(a.hadm_id.map(starts)-a.admit_time).dt.total_seconds()/60.0
    missing=int(a.hospital_start_minute.isna().sum())
    a=a[a.hospital_start_minute.notna()].copy()
    con.register('episode_anchors',a)
    con.execute(f'CREATE OR REPLACE TABLE {d}_anchors AS SELECT * FROM episode_anchors')
    coverage.append(dict(dataset=d,cohort=len(c),episodes=len(onset),
        missing_hospital_start=missing,anchors=len(a)))
direction_json('episode_support.json',coverage)
direction_status('eICU pre-ICU troponin extraction')
mapping=pd.DataFrame([dict(raw_name=k,concept=v) for k,v in EICU_LAB_NAME_MAP.items() if v in ['troponin_t','troponin_i']])
con.register('assay_map',mapping)
lab=verified_source('eICU','lab')
con.execute(f"""CREATE OR REPLACE TABLE eicu_preicu_troponin AS
 SELECT a.stay_id,try_cast(l.labresultoffset AS DOUBLE) event_minute,
 try_cast(l.labresultrevisedoffset AS DOUBLE) available_minute,m.concept,
 lower(trim(coalesce(l.labmeasurenamesystem,l.labmeasurenameinterface,''))) unit,
 try_cast(l.labresult AS DOUBLE) AS value
 FROM read_csv({qs(lab)},header=true,all_varchar=true,sample_size=10000,strict_mode=true) l
 JOIN eicu_anchors a ON try_cast(l.patientunitstayid AS BIGINT)=a.stay_id
 JOIN assay_map m ON lower(trim(l.labname))=m.raw_name
 WHERE try_cast(l.labresultoffset AS DOUBLE)>=greatest(-1440,a.hospital_start_minute)
 AND try_cast(l.labresultoffset AS DOUBLE)<0
 AND try_cast(l.labresult AS DOUBLE) IS NOT NULL""")
direction_status('MIMIC pre-ICU troponin extraction')
root=DRIVE/'Data'/'MIMIC'/'Full'
mapping=pd.DataFrame([dict(itemid=k,concept=v) for k,v in _mimic_lab_item_map(root).items() if v in ['troponin_t','troponin_i']])
con.register('assay_map',mapping)
lab=verified_source('MIMIC','labevents');verified_source('MIMIC','d_labitems')
con.execute(f"""CREATE OR REPLACE TABLE mimic_preicu_troponin AS
 SELECT a.stay_id,date_diff('second',a.admit_time,try_cast(l.charttime AS TIMESTAMP))/60.0 event_minute,
 date_diff('second',a.admit_time,try_cast(l.storetime AS TIMESTAMP))/60.0 available_minute,
 m.concept,lower(trim(coalesce(l.valueuom,''))) unit,try_cast(l.valuenum AS DOUBLE) AS value
 FROM read_csv({qs(lab)},header=true,all_varchar=true,sample_size=10000,strict_mode=true) l
 JOIN mimic_anchors a ON try_cast(l.hadm_id AS BIGINT)=a.hadm_id
 JOIN assay_map m ON try_cast(l.itemid AS BIGINT)=m.itemid
 WHERE date_diff('second',a.admit_time,try_cast(l.charttime AS TIMESTAMP))/60.0>=greatest(-1440,a.hospital_start_minute)
 AND try_cast(l.charttime AS TIMESTAMP)<a.admit_time AND try_cast(l.valuenum AS DOUBLE) IS NOT NULL""")
for key,info in used.items():
    st=Path(info['path']).stat()
    assert st.st_size==info['bytes'] and st.st_mtime_ns==info['mtime_ns'], 'Source changed during scan: '+key
direction_json('source_manifest.json',dict(verified_fingerprints_reused_from=str(PRIOR/'input_manifest.json'),
    same_size_and_mtime_before_and_after=True,sources=used))
direction_status('serial troponin trajectory support')
support=[]
for d in ['eicu','mimic']:
    con.execute(f"""CREATE OR REPLACE TABLE {d}_troponin AS
      WITH combined AS (
        SELECT stay_id,event_minute,available_minute,concept,unit,value FROM {d}_preicu_troponin
        UNION ALL
        SELECT l.stay_id,event_minute,available_minute,concept,unit,value FROM prior.{d}_context_labs l
        JOIN {d}_anchors a USING(stay_id) WHERE concept IN ('troponin_t','troponin_i'))
      SELECT stay_id,event_minute,concept,unit,median(value) AS value,max(available_minute) available_minute
      FROM combined WHERE value>0 AND value<=1000000 GROUP BY stay_id,event_minute,concept,unit""")
    con.execute(f"""CREATE OR REPLACE TABLE {d}_trajectory_support AS
      WITH pre_assays AS (
        SELECT t.stay_id,t.concept,t.unit,count(*) pre_n,min(event_minute) pre_first_time,
        max(event_minute) pre_last_time FROM {d}_troponin t JOIN {d}_anchors a USING(stay_id)
        WHERE event_minute>=a.episode_minute-720 AND event_minute<a.episode_minute
          AND event_minute>=a.hospital_start_minute
        GROUP BY t.stay_id,t.concept,t.unit), chosen AS (
        SELECT * FROM pre_assays QUALIFY row_number() OVER(PARTITION BY stay_id
          ORDER BY pre_first_time,CASE WHEN concept='troponin_t' THEN 0 ELSE 1 END,unit)=1), post_assays AS (
        SELECT c.stay_id,count(t.event_minute) post_n FROM chosen c JOIN {d}_anchors a USING(stay_id)
        LEFT JOIN {d}_troponin t ON t.stay_id=c.stay_id AND t.concept=c.concept AND t.unit=c.unit
          AND t.event_minute>=a.episode_minute+120 AND t.event_minute<=a.episode_minute+720
        GROUP BY c.stay_id)
      SELECT a.stay_id,a.episode_minute,c.concept,c.unit,c.pre_n,c.pre_first_time,c.pre_last_time,
        coalesce(p.post_n,0) post_n,
        coalesce(c.pre_n>=2 AND c.pre_last_time-c.pre_first_time>=60,FALSE) serial_pre,
        coalesce(c.pre_n>=2 AND c.pre_last_time-c.pre_first_time>=60 AND p.post_n>=1,FALSE) complete_trajectory
      FROM {d}_anchors a LEFT JOIN chosen c USING(stay_id) LEFT JOIN post_assays p USING(stay_id)""")
    s=con.execute(f'SELECT * FROM {d}_trajectory_support').df()
    support.append(dict(dataset=d,episodes_with_hospital_boundary=len(s),
        any_pre_episode_assay=int(s.pre_n.notna().sum()),serial_pre_episode=int(s.serial_pre.sum()),
        complete_pre_post_trajectories=int(s.complete_trajectory.sum()),
        passes_count_gate=bool(s.complete_trajectory.sum()>=100)))
direction_json('trajectory_support.json',support)
direction_json('analysis_manifest.json',dict(environment='google_colab' if IN_COLAB else 'local',
    protocol_sha256=DIRECTION_PROTOCOL_SHA256,boundary_amendment_sha256=DIRECTION_BOUNDARY_SHA256,
    completed_utc=datetime.now(timezone.utc).isoformat(),
    all_sources_pass=all(s['passes_count_gate'] for s in support),
    trajectory_directions_inspected=False,mortality_joined=False,synthetic_patients=0,
    biological_discovery=False))
con.close()
print(pd.DataFrame(support).to_string(index=False))
direction_status('biological-ordering feasibility','complete')

episode anchors and hospital boundaries running
eICU pre-ICU troponin extraction running
MIMIC pre-ICU troponin extraction running
serial troponin trajectory support running
dataset  episodes_with_hospital_boundary  any_pre_episode_assay  serial_pre_episode  complete_pre_post_trajectories  passes_count_gate
   eicu                             4309                   1326                 224                             133               True
  mimic                             1398                    288                  40                              23              False
biological-ordering feasibility complete


# Hemoglobin and susceptibility to SpO2-associated myocardial injury

Status: new biological candidate, feasibility protocol only. No hemoglobin
effect estimates, outcome interactions, or therapeutic benefits have been
computed. This is not a claim of novelty or a completed discovery.

## Question and prior evidence

Does hemoglobin measured before ICU SpO2 observation modify the association
between the established instability exposure and subsequent myocardial injury?
The biological rationale is reduced oxygen-carrying capacity: equal saturation
does not imply equal arterial oxygen content. A positive interaction could
motivate a susceptibility hypothesis, but would not measure oxygen delivery,
establish ischemia, or identify a transfusion-responsive population.

This extends the project's existing SpO2–troponin signal. The primary exposure
remains the first-four-hour absolute adjacent change of at least four percentage
points in 15-minute median bins, with positive gaps no longer than 30 minutes,
at least three bins, and two qualifying transitions. No MAP requirement is
added. First-drop and first-rise groups are retained for a later specificity
comparison; a recovery rise is not a valid negative control because it may
follow an unrecorded hypoxic episode.

The broad mechanism is not new. A [2022 COVID-19 cohort](https://pmc.ncbi.nlm.nih.gov/articles/PMC9447453/)
reported an interaction between hemoglobin and SpO2/FiO2 on mortality.
The [MINT HF analysis](https://pmc.ncbi.nlm.nih.gov/articles/PMC11999761)
already studied transfusion strategies in patients with MI, anemia and HF.
Neither is an additional patient dataset for this project. Current literature
searches did not identify the exact ICU SpO2-dynamics interaction, which is
insufficient to establish that it is novel. A replicated regression interaction
alone would not meet the user's paradigm-shifting completion requirement.

## Locked feasibility extraction

Use only the canonical MIMIC and eICU HF cohorts and raw sources in Drive Data.
Use the existing production-quality SpO2 bins, preserving their raw floating
point timestamps. Measurements at ICU minute 240 are excluded.

1. Extract explicitly identified hemoglobin laboratory measurements from ICU
   minutes -1440 through 0 inclusive. Measurements must belong to the same
   hospital encounter. MIMIC requires matching hadm_id and a charttime no earlier
   than the earlier of ED registration and inpatient admission. eICU requires
   matching patientunitstayid and a labresultoffset no earlier than
   hospitaladmitoffset. Do not assign unlinked MIMIC laboratories by proximity.
   The supplied MIMIC dictionary identifies item 51222 as Blood/Hematology
   Hemoglobin; use this item only, excluding blood-gas item 50811 and chemistry
   variants. For eICU, require labtypeid 3 and a trimmed case-insensitive exact
   name of Hgb, Hemoglobin or Haemoglobin. The [official table documentation](https://eicu.mit.edu/eicutables/lab/)
   identifies type 3 as hematology and type 7 as ABG. Audit all source names
   containing hgb or hemoglobin before applying this explicit whitelist.
2. Resolve hemoglobin source names and units from the source dictionary and
   aggregate laboratory names/units before deriving hemoglobin strata. Prefer
   hematology/CBC hemoglobin to a blood-gas estimate. Never substitute hematocrit
   divided by three. Preserve original values, units and source identifiers.
3. Normalize documented g/L to g/dL by division by ten and documented g/dL
   without conversion. Unknown or ambiguous units fail closed. Retain a broad
   plausibility range of 3–22 g/dL; report excluded counts. This is a quality
   rule, not an outcome-optimized threshold.
4. Select the latest pre-ICU CBC value, with same-time duplicates collapsed by
   median. If a database has no CBC-specific designation, document the source
   limitation before any outcome analysis. Do not select a value based on
   its magnitude or use post-ICU hemoglobin to improve coverage.
5. Produce aggregate coverage by database and established SpO2 exposure group,
   and availability of the existing assay-matched pre/post troponin endpoint.
   Do not expose mortality counts or troponin-rise frequencies across hemoglobin
   values or strata in this phase. Counts of observed endpoints are permitted;
   their outcomes are not used to tune eligibility.

## Feasibility decision

The candidate proceeds to an analysis specification only if each database has
at least 200 patients with pre-ICU hemoglobin and an observed 12-hour troponin
endpoint, including at least 50 with and 50 without the established SpO2
instability exposure. This is a minimal estimability screen, not a power claim.
An interaction power/precision assessment and a frozen confounder model are
still required before opening hemoglobin-by-outcome associations. Failure
closes this specification; do not substitute a post-ICU hemoglobin or enlarge
the exposure window to rescue it.

## Required next-stage protections

Before fitting an outcome model, freeze the estimand, continuous hemoglobin
functional form, interaction scale, multiplicity family, missing-data strategy,
support diagnostics and effect size needed to justify further investigation.
Explicitly address absolute saturation, hypoxemic burden, sampling, renal
function, circulatory state, respiratory support and vasoactive treatment.
Incomplete pressure/treatment data and transfusion/bleeding confounding must
not be hidden by a statistically significant coefficient. Discharge diagnosis
codes must not be represented as prospectively available covariates.

Use hospital mortality only as a separately specified clinical-relevance
endpoint after the model lock. No causal treatment or mortality-reduction
claim follows from this observational interaction. MIMIC and eICU have both
been explored already and are not pristine discovery/validation holdouts.

Patient-level outputs remain outside git in Drive Data. The eventual executable
workflow belongs in the single primary Colab notebook. All actual execution
must be labeled local unless it truly ran in Colab. The prior unsupported
circulatory-context and troponin-ordering experiments remain visible.

In [12]:
HEMOGLOBIN_PROTOCOL_SHA256='e983447d8971cb9a8c5b633be8cb4e5934b7f7a0b6909174f51b749007870628'

In [13]:
# Pre-ICU hemoglobin feasibility only; no outcome interactions are computed.
import sys,json,importlib.util,hashlib
from pathlib import Path
from datetime import datetime,timezone
import pandas as pd
import numpy as np
import duckdb
IN_COLAB=importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
DRIVE=Path('/content/drive/MyDrive') if IN_COLAB else Path.home()/'Library/CloudStorage/GoogleDrive-2arnavmana@gmail.com/My Drive'
PROJECT=DRIVE/'Projects'/'PhysioGraph'
PRIVATE=DRIVE/'Data'/'PhysioGraph_Biological_Discovery_20260905'
OUT=PROJECT/'research'/'spo2_hemoglobin';OUT.mkdir(exist_ok=True)
PRIOR=PROJECT/'research'/'spo2_circulatory_context'
def hb_json(name,obj):
    (OUT/name).write_text(json.dumps(obj,indent=2,allow_nan=False,default=str)+'\n')
def hb_status(stage,status='running'):
    hb_json('run_status.json',dict(stage=stage,status=status,
        environment='google_colab' if IN_COLAB else 'local',
        updated_utc=datetime.now(timezone.utc).isoformat(),goal_complete=False))
    print(stage,status,flush=True)
lock=OUT/'protocol_lock.json'
if lock.exists():assert json.loads(lock.read_text())['sha256']==HEMOGLOBIN_PROTOCOL_SHA256
else:hb_json('protocol_lock.json',dict(sha256=HEMOGLOBIN_PROTOCOL_SHA256,
    locked_before_hemoglobin_extraction=True,outcome_interactions_not_seen=True,
    utc=datetime.now(timezone.utc).isoformat()))
con=duckdb.connect(str(PRIVATE/'spo2_hemoglobin.duckdb'))
con.execute('SET threads=2');con.execute("SET memory_limit='2GB'")
def qs(value):return "'"+str(value).replace("'","''")+"'"
con.execute(f"ATTACH {qs(PRIVATE/'spo2_context.duckdb')} AS prior (READ_ONLY)")
manifest=json.loads((PRIOR/'input_manifest.json').read_text());used={}
def hb_source(dataset,name):
    key=dataset+'/'+name;info=manifest[key];p=Path(info['path']);s=p.stat()
    assert s.st_size==info['bytes'] and s.st_mtime_ns==info['mtime_ns'],key
    used[key]=info
    return p
hb_status('pre-ICU hemoglobin source extraction')
for d,source in [('eicu','eICU'),('mimic','MIMIC')]:
    extra=',c.admit_time,c.hadm_id' if d=='mimic' else ''
    eligible=con.execute(f"""SELECT c.stay_id,f.spo2_jump_any AS exposed,
        e.troponin_rise_12h IS NOT NULL AS troponin_observed {extra}
        FROM prior.{d}_cohort c JOIN prior.{d}_features f USING(stay_id)
        JOIN prior.{d}_context_endpoints e USING(stay_id)
        WHERE f.spo2_bins>=3 AND f.spo2_transitions>=2""").df()
    if d=='eicu':
        patient=pd.read_csv(hb_source(source,'patient'),
            usecols=['patientunitstayid','hospitaladmitoffset']).set_index('patientunitstayid')
        eligible['hospital_start_minute']=pd.to_numeric(
            eligible.stay_id.map(patient.hospitaladmitoffset),errors='coerce')
    else:
        admission=pd.read_csv(hb_source(source,'admissions'),
            usecols=['hadm_id','admittime','edregtime']).set_index('hadm_id')
        start=admission.apply(pd.to_datetime,errors='coerce').min(axis=1)
        eligible['hospital_start_minute']=(eligible.hadm_id.map(start)-eligible.admit_time).dt.total_seconds()/60
        dictionary=pd.read_csv(hb_source(source,'d_labitems'))
        selected=dictionary[dictionary.itemid.eq(51222)]
        assert len(selected)==1 and selected.iloc[0]['label']=='Hemoglobin'
        assert selected.iloc[0]['category']=='Hematology' and selected.iloc[0]['fluid']=='Blood'
    con.register('eligible_frame',eligible)
    con.execute(f'CREATE OR REPLACE TABLE {d}_eligible AS SELECT * FROM eligible_frame')
    con.execute(f"""CREATE OR REPLACE TABLE {d}_first_direction AS
      WITH b AS (SELECT stay_id,bin,value,representative_minute,
        value-lag(value) OVER(PARTITION BY stay_id ORDER BY bin) delta,
        representative_minute-lag(representative_minute) OVER(PARTITION BY stay_id ORDER BY bin) gap
        FROM prior.{d}_bins WHERE concept='spo2'),
      first_jump AS (SELECT stay_id,delta FROM b WHERE abs(delta)>=4 AND gap>0 AND gap<=30
        QUALIFY row_number() OVER(PARTITION BY stay_id ORDER BY bin)=1)
      SELECT e.stay_id,CASE WHEN j.delta<0 THEN 'drop' WHEN j.delta>0 THEN 'rise' ELSE 'none' END first_direction
      FROM {d}_eligible e LEFT JOIN first_jump j USING(stay_id)""")
    assert con.execute(f"""SELECT count(*) FROM {d}_eligible e JOIN {d}_first_direction f USING(stay_id)
      WHERE (e.exposed=1)<>(f.first_direction<>'none')""").fetchone()[0]==0
    lab=hb_source(source,'lab' if d=='eicu' else 'labevents')
    if d=='eicu':
        con.execute(f"""CREATE OR REPLACE TABLE eicu_hb_raw AS
          SELECT e.stay_id,try_cast(l.labresultoffset AS DOUBLE) event_minute,
            try_cast(l.labresultrevisedoffset AS DOUBLE) available_minute,
            l.labname raw_name,try_cast(l.labtypeid AS INTEGER) lab_type,
            l.labmeasurenamesystem unit_system,l.labmeasurenameinterface unit_interface,
            try_cast(l.labresult AS DOUBLE) raw_value,
            lower(trim(l.labname)) IN ('hgb','hemoglobin','haemoglobin')
              AND try_cast(l.labtypeid AS INTEGER)=3 AS source_approved
          FROM read_csv({qs(lab)},header=true,all_varchar=true,sample_size=10000,strict_mode=true) l
          JOIN eicu_eligible e ON try_cast(l.patientunitstayid AS BIGINT)=e.stay_id
          WHERE (lower(l.labname) LIKE '%hgb%' OR lower(l.labname) LIKE '%hemoglobin%'
            OR lower(l.labname) LIKE '%haemoglobin%')
          AND try_cast(l.labresultoffset AS DOUBLE) BETWEEN greatest(-1440,e.hospital_start_minute) AND 0
          AND e.hospital_start_minute IS NOT NULL""")
    else:
        con.execute(f"""CREATE OR REPLACE TABLE mimic_hb_raw AS
          SELECT e.stay_id,date_diff('second',e.admit_time,try_cast(l.charttime AS TIMESTAMP))/60.0 event_minute,
            date_diff('second',e.admit_time,try_cast(l.storetime AS TIMESTAMP))/60.0 available_minute,
            'Hemoglobin' raw_name,3 AS lab_type,l.valueuom unit_system,'' unit_interface,
            try_cast(l.valuenum AS DOUBLE) raw_value,TRUE AS source_approved
          FROM read_csv({qs(lab)},header=true,all_varchar=true,sample_size=10000,strict_mode=true) l
          JOIN mimic_eligible e ON try_cast(l.hadm_id AS BIGINT)=e.hadm_id
          WHERE try_cast(l.itemid AS INTEGER)=51222
          AND date_diff('second',e.admit_time,try_cast(l.charttime AS TIMESTAMP))/60.0
            BETWEEN greatest(-1440,e.hospital_start_minute) AND 0
          AND e.hospital_start_minute IS NOT NULL""")
for key,info in used.items():
    s=Path(info['path']).stat()
    assert s.st_size==info['bytes'] and s.st_mtime_ns==info['mtime_ns'],key
audit=[]
for d in ['eicu','mimic']:
    rows=con.execute(f"""SELECT raw_name,lab_type,unit_system,unit_interface,source_approved,count(*) records
      FROM {d}_hb_raw GROUP BY ALL ORDER BY records DESC""").df()
    rows['dataset']=d;rows=rows.astype(object).where(pd.notna(rows),None);audit.extend(rows.to_dict('records'))
hb_json('source_name_unit_audit.json',audit)
# Source name/type whitelist and accepted units were fixed before these values were normalized.
hb_status('hemoglobin normalization and endpoint availability')
coverage=[];quality=[];validation=[]
for d in ['eicu','mimic']:
    raw=con.execute(f'SELECT * FROM {d}_hb_raw').df()
    for key in ['unit_system','unit_interface']:
        raw[key+'_clean']=raw[key].fillna('').str.strip().str.lower().str.replace(' ','',regex=False)
    u=raw.unit_system_clean.where(raw.unit_system_clean.ne(''),raw.unit_interface_clean)
    conflict=raw.unit_system_clean.ne('')&raw.unit_interface_clean.ne('')&raw.unit_system_clean.ne(raw.unit_interface_clean)
    raw['unit_approved']=u.isin(['g/dl','g/l'])&~conflict
    raw['hemoglobin']=raw.raw_value/np.where(u.eq('g/l'),10.0,1.0)
    raw['quality_approved']=raw.source_approved.fillna(False)&raw.unit_approved&raw.hemoglobin.between(3,22)
    quality.append(dict(dataset=d,candidate_records=len(raw),
        wrong_source=int((~raw.source_approved.fillna(False)).sum()),
        unknown_or_conflicting_unit=int((~raw.unit_approved).sum()),
        outside_value_range_or_missing=int((~raw.hemoglobin.between(3,22)).sum()),
        accepted_records=int(raw.quality_approved.sum())))
    con.register('normalized_frame',raw)
    con.execute(f'CREATE OR REPLACE TABLE {d}_hb_normalized AS SELECT * FROM normalized_frame')
    con.execute(f"""CREATE OR REPLACE TABLE {d}_baseline_hb AS
      SELECT stay_id,event_minute,median(hemoglobin) hemoglobin,max(available_minute) available_minute
      FROM {d}_hb_normalized WHERE quality_approved GROUP BY stay_id,event_minute
      QUALIFY row_number() OVER(PARTITION BY stay_id ORDER BY event_minute DESC)=1""")
    independent=raw[raw.quality_approved].groupby(['stay_id','event_minute'],as_index=False).hemoglobin.median()
    independent=independent.sort_values('event_minute').drop_duplicates('stay_id',keep='last').set_index('stay_id')
    baseline=con.execute(f'SELECT * FROM {d}_baseline_hb').df().set_index('stay_id')
    assert set(independent.index)==set(baseline.index)
    np.testing.assert_allclose(independent.hemoglobin.sort_index(),baseline.hemoglobin.sort_index(),rtol=0,atol=1e-12)
    validation.append(dict(dataset=d,independent_latest_value_match=True,people=len(baseline)))
    support=con.execute(f"""SELECT e.exposed,count(*) eligible,count(b.hemoglobin) pre_icu_hemoglobin,
      sum(CASE WHEN b.hemoglobin IS NOT NULL AND e.troponin_observed THEN 1 ELSE 0 END) paired_endpoint_observed,
      sum(CASE WHEN b.hemoglobin IS NOT NULL AND b.available_minute>0 THEN 1 ELSE 0 END) result_entered_after_icu_admission
      FROM {d}_eligible e LEFT JOIN {d}_baseline_hb b USING(stay_id) GROUP BY e.exposed ORDER BY e.exposed""").df()
    observed={int(r.exposed):int(r.paired_endpoint_observed) for r in support.itertuples()}
    passes=sum(observed.values())>=200 and observed.get(0,0)>=50 and observed.get(1,0)>=50
    coverage.append(dict(dataset=d,passes_minimal_feasibility=passes,groups=support.to_dict('records')))
hb_json('coverage.json',coverage);hb_json('quality_audit.json',quality)
hb_json('independent_validation.json',validation)
hb_json('source_manifest.json',dict(verified_fingerprints_reused_from=str(PRIOR/'input_manifest.json'),
    same_size_and_mtime_before_and_after=True,sources=used))
hb_json('analysis_manifest.json',dict(protocol_sha256=HEMOGLOBIN_PROTOCOL_SHA256,
    builder_sha256=hashlib.sha256((PROJECT/'scripts/build_biological_notebook.py').read_bytes()).hexdigest(),
    environment='google_colab' if IN_COLAB else 'local',completed_utc=datetime.now(timezone.utc).isoformat(),
    all_sources_pass=all(s['passes_minimal_feasibility'] for s in coverage),
    outcome_interactions_inspected=False,mortality_joined=False,synthetic_patients=0,biological_discovery=False))
con.close()
print(json.dumps(coverage,indent=2))
hb_status('hemoglobin susceptibility feasibility','complete')

pre-ICU hemoglobin source extraction running
hemoglobin normalization and endpoint availability running
[
  {
    "dataset": "eicu",
    "passes_minimal_feasibility": true,
    "groups": [
      {
        "exposed": 0,
        "eligible": 7143,
        "pre_icu_hemoglobin": 2445,
        "paired_endpoint_observed": 409.0,
        "result_entered_after_icu_admission": 151.0
      },
      {
        "exposed": 1,
        "eligible": 4309,
        "pre_icu_hemoglobin": 1539,
        "paired_endpoint_observed": 250.0,
        "result_entered_after_icu_admission": 69.0
      }
    ]
  },
  {
    "dataset": "mimic",
    "passes_minimal_feasibility": false,
    "groups": [
      {
        "exposed": 0,
        "eligible": 3313,
        "pre_icu_hemoglobin": 774,
        "paired_endpoint_observed": 80.0,
        "result_entered_after_icu_admission": 60.0
      },
      {
        "exposed": 1,
        "eligible": 1398,
        "pre_icu_hemoglobin": 339,
        "paired_endpoint_observed": 53.0,

# Pre-ICU laboratory encounter-linkage audit

This is an outcome-blind source audit needed to determine whether the current
SpO2 biological experiments are losing relevant pre-ICU observations. It is
not a biological finding and does not fulfill the discovery goal.

The linked-record hemoglobin specification completed with 659 usable eICU and
133 MIMIC observations. MIMIC had 1,441 candidate CBC records; none was excluded
for units and only three for the frozen value range. Thus unit handling does
not explain its low coverage. The earlier troponin-ordering specification also
had sparse MIMIC pre-episode measurements. Neither specification is changed
by this audit, and their failed support results remain visible.

The [official MIMIC laboratory documentation](https://mimic.mit.edu/docs/iv/modules/hosp/labevents.html)
states that some temporally associated hospital laboratories lack hadm_id and
that time-based joins may be needed. The following fixed audit evaluates that
possibility; it does not assume all unlinked tests belong to an admission.

## Extraction

Use the supplied MIMIC files only. Restrict patient windows to the existing
SpO2-dynamics-eligible HF cohort, with laboratory charttime in the 24 hours
before ICU admission through ICU minute zero. Extract identifiers, specimen
identifier, item, timestamps, units, and whether a positive numeric result
exists. Do not select laboratory concentrations or any mortality outcomes.

Include CBC hemoglobin item 51222 and the existing production troponin item
mapping. Items 50811 and 51640 are included only to inventory alternative
hemoglobin sources; they are not accepted into the frozen CBC experiment.
No oxygen, troponin or hemoglobin threshold will be optimized in this audit.

## Fixed temporal rule and validation

For each candidate laboratory, use the same subject_id and the complete
admissions table, including admissions outside the HF cohort. A temporal
candidate requires charttime between the earlier of edregtime and admittime,
and dischtime, with both boundaries valid. No additional time tolerance,
nearest-admission assignment or outpatient proximity rule is allowed.

Count zero, one and multiple matching hospital encounters. A missing hadm_id
can only be a candidate for later use when exactly one encounter matches.
Never overwrite an explicit nonmissing hadm_id, including when it conflicts
with a temporal candidate. Preserve candidate assignments separately from
all prior patient tables and the raw sources.

Validate the fixed rule by masking known hadm_id values: compute the proportion
of uniquely temporally assigned records for which the candidate exactly equals
the recorded identifier, separately for CBC, troponin and alternative Hb.
Report coverage, errors, ambiguous assignments and confidence intervals rather
than accuracy alone. To avoid treating repeated tests from one patient as
independent validation, define patient success as all of that patient's
uniquely assigned known records agreeing with their recorded IDs. A candidate
procedure requires at least 200 distinct patients for each analyte family it
would support and a one-sided 95% exact lower confidence bound above 99% for
this patient-level agreement. Also report the record-level agreement descriptively.
This is a necessary validation condition, not proof that missing links have
the same error distribution as known links. Do not tune the rule to pass it.

Report the number of eligible encounters with additional uniquely assignable
missing-ID observations, distinguishing records before formal inpatient
admission from those after it. Do not recompute troponin trajectories, Hb
interaction effects, mortality associations or biological support gates.

## Decision boundary

Any use of newly assigned records requires a separately documented amendment
that acknowledges the earlier feasibility results and the original explicit-ID
restriction. The original plans and results must remain unchanged. Passing
this audit would justify considering an amendment, not silently reclassifying
the original experiments as successful or preregistered under the new rule.

Patient-level records remain in a separate private database in Drive Data.
Aggregate audit results and provenance are saved with the primary notebook.
Execution is local unless actual Colab execution is documented.

In [14]:
LINKAGE_PROTOCOL_SHA256='ab5b34e82ca7af1ed34fdbd6acd616c95f1aa07a7a3598cafbe3719e841d5c7a'

In [15]:
# Outcome-blind MIMIC encounter-linkage audit. No concentrations are selected.
import sys,json,hashlib,importlib.util
from pathlib import Path
from datetime import datetime,timezone
import pandas as pd
import duckdb
from scipy.stats import beta
IN_COLAB=importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
DRIVE=Path('/content/drive/MyDrive') if IN_COLAB else Path.home()/'Library/CloudStorage/GoogleDrive-2arnavmana@gmail.com/My Drive'
PROJECT=DRIVE/'Projects'/'PhysioGraph'
PRIVATE=DRIVE/'Data'/'PhysioGraph_Biological_Discovery_20260905'
OUT=PROJECT/'research'/'mimic_preicu_lab_linkage';OUT.mkdir(exist_ok=True)
PRIOR=PROJECT/'research'/'spo2_circulatory_context'
sys.path.insert(0,str(PROJECT/'src'))
from physiograph.etl.mimic_extractor import _mimic_lab_item_map
def link_json(name,obj):
    (OUT/name).write_text(json.dumps(obj,indent=2,allow_nan=False,default=str)+'\n')
def link_status(stage,status='running'):
    link_json('run_status.json',dict(stage=stage,status=status,
        environment='google_colab' if IN_COLAB else 'local',
        updated_utc=datetime.now(timezone.utc).isoformat(),goal_complete=False))
    print(stage,status,flush=True)
lock=OUT/'protocol_lock.json'
if lock.exists():assert json.loads(lock.read_text())['sha256']==LINKAGE_PROTOCOL_SHA256
else:link_json('protocol_lock.json',dict(sha256=LINKAGE_PROTOCOL_SHA256,
    locked_before_linkage_audit=True,previous_biological_support_counts_seen=True,
    outcome_interactions_not_seen=True,utc=datetime.now(timezone.utc).isoformat()))
manifest=json.loads((PRIOR/'input_manifest.json').read_text());used={}
def link_source(name):
    key='MIMIC/'+name;info=manifest[key];p=Path(info['path']);s=p.stat()
    assert s.st_size==info['bytes'] and s.st_mtime_ns==info['mtime_ns'],key
    used[key]=info
    return p
def qs(value):return "'"+str(value).replace("'","''")+"'"
con=duckdb.connect(str(PRIVATE/'mimic_preicu_lab_linkage.duckdb'))
con.execute('SET threads=2');con.execute("SET memory_limit='2GB'")
con.execute(f"ATTACH {qs(PRIVATE/'spo2_context.duckdb')} AS prior (READ_ONLY)")
con.execute("""CREATE OR REPLACE TABLE eligible AS
  SELECT c.stay_id,c.person_id AS subject_id,c.hadm_id,c.admit_time
  FROM prior.mimic_cohort c JOIN prior.mimic_features f USING(stay_id)
  WHERE f.spo2_bins>=3 AND f.spo2_transitions>=2""")
admission=pd.read_csv(link_source('admissions'),
    usecols=['subject_id','hadm_id','admittime','edregtime','dischtime'])
assert admission.hadm_id.is_unique
for key in ['admittime','edregtime','dischtime']:
    admission[key]=pd.to_datetime(admission[key],errors='coerce')
admission['encounter_start']=admission[['admittime','edregtime']].min(axis=1)
admission['valid_boundary']=admission.encounter_start.notna()&admission.dischtime.notna()&admission.dischtime.ge(admission.encounter_start)
con.register('admission_frame',admission)
con.execute('CREATE OR REPLACE TABLE admissions AS SELECT * FROM admission_frame')
assert con.execute("""SELECT count(*) FROM eligible e JOIN admissions a USING(hadm_id)
    WHERE e.subject_id<>a.subject_id""").fetchone()[0]==0
link_source('d_labitems')
mapping={k:'troponin' for k,v in _mimic_lab_item_map(DRIVE/'Data/MIMIC/Full').items() if v in ['troponin_i','troponin_t']}
mapping.update({51222:'cbc_hemoglobin',50811:'blood_gas_hemoglobin',51640:'chemistry_hemoglobin'})
itemmap=pd.DataFrame([dict(itemid=k,family=v) for k,v in mapping.items()])
con.register('item_map',itemmap)
lab=link_source('labevents')
link_status('pre-ICU laboratory identifier scan')
con.execute(f"""CREATE OR REPLACE TABLE candidate_labs AS
 SELECT try_cast(l.labevent_id AS BIGINT) labevent_id,try_cast(l.subject_id AS BIGINT) subject_id,
   try_cast(l.hadm_id AS BIGINT) recorded_hadm_id,coalesce(trim(l.hadm_id),'')='' AS missing_hadm_id,
   try_cast(l.specimen_id AS BIGINT) specimen_id,try_cast(l.itemid AS INTEGER) itemid,m.family,
   try_cast(l.charttime AS TIMESTAMP) charttime,try_cast(l.storetime AS TIMESTAMP) storetime,
   l.valueuom unit,coalesce(try_cast(l.valuenum AS DOUBLE)>0,FALSE) positive_numeric_available
 FROM read_csv({qs(lab)},header=true,all_varchar=true,sample_size=10000,strict_mode=true) l
 JOIN item_map m ON try_cast(l.itemid AS INTEGER)=m.itemid
 WHERE EXISTS(SELECT 1 FROM eligible e WHERE e.subject_id=try_cast(l.subject_id AS BIGINT)
   AND try_cast(l.charttime AS TIMESTAMP) BETWEEN e.admit_time-INTERVAL '24 hours' AND e.admit_time)""")
assert con.execute('SELECT count(*)=count(DISTINCT labevent_id) FROM candidate_labs').fetchone()[0]
for key,info in used.items():
    s=Path(info['path']).stat()
    assert s.st_size==info['bytes'] and s.st_mtime_ns==info['mtime_ns'],key
link_status('fixed temporal assignment and known-ID validation')
con.execute("""CREATE OR REPLACE TABLE temporal_candidates AS
 SELECT l.labevent_id,a.hadm_id AS candidate_hadm_id,a.admittime
 FROM candidate_labs l JOIN admissions a ON l.subject_id=a.subject_id
   AND a.valid_boundary AND l.charttime BETWEEN a.encounter_start AND a.dischtime""")
con.execute("""CREATE OR REPLACE TABLE linkage AS
 SELECT l.*,coalesce(t.candidates,0) temporal_candidate_count,
   CASE WHEN t.candidates=1 THEN t.candidate_hadm_id ELSE NULL END unique_temporal_hadm_id
 FROM candidate_labs l LEFT JOIN (
   SELECT labevent_id,count(*) candidates,min(candidate_hadm_id) candidate_hadm_id
   FROM temporal_candidates GROUP BY labevent_id) t USING(labevent_id)""")
validation=[];inventory=[]
for family in itemmap.family.unique():
    counts=con.execute("""SELECT count(*) records,
      count(*) FILTER(WHERE recorded_hadm_id IS NOT NULL) known_id,
      count(*) FILTER(WHERE missing_hadm_id) missing_id,
      count(*) FILTER(WHERE NOT missing_hadm_id AND recorded_hadm_id IS NULL) invalid_id,
      count(*) FILTER(WHERE temporal_candidate_count=0) no_temporal_match,
      count(*) FILTER(WHERE temporal_candidate_count>1) ambiguous_temporal_match,
      count(*) FILTER(WHERE recorded_hadm_id IS NOT NULL AND temporal_candidate_count=1) uniquely_assigned_known,
      count(*) FILTER(WHERE recorded_hadm_id IS NOT NULL AND temporal_candidate_count=1
        AND recorded_hadm_id=unique_temporal_hadm_id) correctly_assigned_known,
      count(*) FILTER(WHERE missing_hadm_id AND temporal_candidate_count=1) uniquely_assigned_missing
      FROM linkage WHERE family=?""",[family]).df().iloc[0].to_dict()
    counts={k:int(v) for k,v in counts.items()}
    n=counts['uniquely_assigned_known'];k=counts['correctly_assigned_known']
    patients,correct_patients=con.execute("""SELECT count(*),coalesce(sum(all_correct),0) FROM (
      SELECT subject_id,bool_and(recorded_hadm_id=unique_temporal_hadm_id) all_correct
      FROM linkage WHERE family=? AND recorded_hadm_id IS NOT NULL AND temporal_candidate_count=1
      GROUP BY subject_id)""",[family]).fetchone()
    patients=int(patients);correct_patients=int(correct_patients)
    lower=float(beta.ppf(.05,correct_patients,patients-correct_patients+1)) if correct_patients else (0.0 if patients else None)
    validation.append(dict(family=family,**counts,
      known_id_accuracy=k/n if n else None,validation_patients=patients,all_correct_patients=correct_patients,
      patient_agreement_one_sided_95pct_lower=lower,
      passes_validation=bool(patients>=200 and lower is not None and lower>.99)))
inventory=con.execute("""SELECT family,itemid,unit,missing_hadm_id,count(*) records,
  count(*) FILTER(WHERE positive_numeric_available) positive_numeric_records
  FROM linkage GROUP BY ALL ORDER BY family,itemid,unit""").df()
inventory=inventory.astype(object).where(pd.notna(inventory),None).to_dict('records')
con.execute("""CREATE OR REPLACE TABLE eligible_candidate_additions AS
 SELECT e.stay_id,l.labevent_id,l.family,l.unique_temporal_hadm_id AS candidate_hadm_id,
   l.charttime<a.admittime AS before_inpatient_admission,l.positive_numeric_available
 FROM linkage l JOIN eligible e ON l.unique_temporal_hadm_id=e.hadm_id
   AND l.subject_id=e.subject_id AND l.charttime BETWEEN e.admit_time-INTERVAL '24 hours' AND e.admit_time
 JOIN admissions a ON a.hadm_id=e.hadm_id
 WHERE l.missing_hadm_id AND l.temporal_candidate_count=1""")
additions=con.execute("""SELECT family,count(*) candidate_records,count(DISTINCT stay_id) eligible_encounters,
 count(*) FILTER(WHERE before_inpatient_admission) records_before_inpatient_admission,
 count(DISTINCT stay_id) FILTER(WHERE before_inpatient_admission) encounters_with_pre_inpatient_records
 FROM eligible_candidate_additions GROUP BY family ORDER BY family""").df().to_dict('records')
link_json('known_id_validation.json',validation)
link_json('source_inventory.json',inventory)
link_json('candidate_additions.json',additions)
link_json('admission_boundaries.json',dict(admissions=len(admission),
    invalid_boundaries=int((~admission.valid_boundary).sum()),eligible_encounters=con.execute('SELECT count(*) FROM eligible').fetchone()[0]))
link_json('source_manifest.json',dict(verified_fingerprints_reused_from=str(PRIOR/'input_manifest.json'),
    same_size_and_mtime_before_and_after=True,sources=used))
link_json('analysis_manifest.json',dict(protocol_sha256=LINKAGE_PROTOCOL_SHA256,
    builder_sha256=hashlib.sha256((PROJECT/'scripts/build_biological_notebook.py').read_bytes()).hexdigest(),
    environment='google_colab' if IN_COLAB else 'local',completed_utc=datetime.now(timezone.utc).isoformat(),
    lab_concentrations_selected=False,mortality_joined=False,prior_patient_tables_modified=False,
    biological_discovery=False,amendment_applied=False))
con.close()
print(json.dumps(dict(validation=validation,candidate_additions=additions),indent=2))
link_status('MIMIC pre-ICU linkage audit','complete')

pre-ICU laboratory identifier scan running
fixed temporal assignment and known-ID validation running
{
  "validation": [
    {
      "family": "troponin",
      "records": 2834,
      "known_id": 741,
      "missing_id": 2093,
      "invalid_id": 0,
      "no_temporal_match": 86,
      "ambiguous_temporal_match": 4,
      "uniquely_assigned_known": 735,
      "correctly_assigned_known": 735,
      "uniquely_assigned_missing": 2009,
      "known_id_accuracy": 1.0,
      "validation_patients": 555,
      "all_correct_patients": 555,
      "patient_agreement_one_sided_95pct_lower": 0.9946168256843885,
      "passes_validation": true
    },
    {
      "family": "cbc_hemoglobin",
      "records": 4385,
      "known_id": 1479,
      "missing_id": 2906,
      "invalid_id": 0,
      "no_temporal_match": 250,
      "ambiguous_temporal_match": 1,
      "uniquely_assigned_known": 1445,
      "correctly_assigned_known": 1445,
      "uniquely_assigned_missing": 2689,
      "known_id_accuracy": 1.0

# Amendment: validated temporal linkage for pre-ICU laboratories

This amendment follows, and does not erase, the failed explicit-ID-only
feasibility results: MIMIC had 23 complete injury-ordering trajectories and
133 Hb-plus-troponin observations. Biological trajectory directions and
hemoglobin-by-outcome associations have not been inspected.

## Evidence supporting the amendment

The fixed, outcome-blind audit used subject_id and documented encounter bounds
against all 546,028 supplied MIMIC admissions. It found 2,681 additional CBC
and 2,009 troponin records uniquely assignable to eligible encounters. Most
preceded formal inpatient admission. It did not select lab concentrations.

Among uniquely assigned records with known IDs, all assignments agreed for
1,091 CBC patients and 555 troponin patients. The one-sided 95% exact lower
bounds for patient-level complete agreement were 99.73% and 99.46%, exceeding
the frozen 99% requirement. An independent Pandas implementation reproduced
all candidate and ambiguity assignments for all 9,175 audited records.
See `research/mimic_preicu_lab_linkage/` for the counts and provenance.

The [MIMIC documentation](https://mimic.mit.edu/docs/iv/modules/hosp/labevents.html)
explicitly warns that some temporally associated labs lack hadm_id and
describes time-based joins as an option. Validation on known IDs does not
prove that missing-ID records have the same error distribution; the latter
remain inferred links and must be labeled as such.

## Sole analytical change

Permit MIMIC CBC and troponin records with a genuinely missing hadm_id when
the already-frozen audit assigns exactly one hospital encounter for the same
subject within its documented ED/inpatient-to-discharge bounds. The record
must be in the audited 24-hour pre-ICU window through ICU minute zero and
assigned to the same canonical eligible hospital encounter. Use the saved
candidate lab IDs; do not optimize the matching rule or include additional
records found by a wider search.

Never overwrite a nonmissing recorded ID or include an ambiguous assignment.
Keep source_hadm_id and assigned_hadm_id distinct in the restored-record table.
Record linkage provenance and retain the original explicit-ID analyses.
Do not modify the raw source files or the prior patient tables.

No additional hemoglobin assay is introduced: CBC item 51222 remains the sole
MIMIC Hb source. Blood-gas and chemistry Hb remain excluded. Hemoglobin units,
3–22 g/dL quality range, pre-ICU latest-value selection, SpO2 definitions,
troponin assay matching and positivity bounds, outcome horizons, serial-sample
separation, and both feasibility thresholds are unchanged. eICU is unchanged.
The Hb experiment continues to use the existing ICU-baseline troponin endpoint;
this amendment does not expand that endpoint's baseline window.

## Reruns and decision

1. Retrieve actual concentrations only for the saved, eligible missing-ID CBC
   and troponin records. Verify source fingerprints and exact lab-ID coverage.
2. Recompute the Hb availability screen using the original normalization and
   latest-value algorithm, adding only the restored CBC records.
3. Recompute the original injury-ordering availability screen, adding only the
   restored troponin records to the original episode anchors and assay rules.
4. Report original and amended counts together. These are two analyses of
   overlapping data, not independent replications. An amended pass permits a
   separately frozen biological analysis; it is not evidence for the hypothesis.

Do not open Hb-by-outcome associations or troponin trajectory directions in
these reruns. If a feasibility gate still fails, report that failure without
lowering its threshold or enlarging its biological windows. Later analyses
must preserve explicit-ID-only sensitivity estimates and linkage uncertainty.

The primary deliverable remains one notebook using MIMIC and eICU. All current
execution is local. A linkage improvement is not the requested biological
discovery and cannot establish a mortality benefit.

In [16]:
AMENDMENT_PROTOCOL_SHA256='bbbcd2db065fe5a8c01d6d5a0739c6f3c6d46f5fd93608b7a7fa5993985f8ed9'
HEMOGLOBIN_PROTOCOL_SHA256='e983447d8971cb9a8c5b633be8cb4e5934b7f7a0b6909174f51b749007870628'
DIRECTION_PROTOCOL_SHA256='cb137c6444593ddfa99eff90fd85c0216f92750c33f8f9c0d25290774f140490'
DIRECTION_BOUNDARY_SHA256='e9105d2d2973387c0181882f02e006d284d297a70efa73f4d34dc5b6b3e1fc2b'

In [17]:
# Restore only the saved uniquely assigned missing-ID records in a separate database.
import sys,json,hashlib,importlib.util
from pathlib import Path
from datetime import datetime,timezone
import pandas as pd
import numpy as np
import duckdb
IN_COLAB=importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
DRIVE=Path('/content/drive/MyDrive') if IN_COLAB else Path.home()/'Library/CloudStorage/GoogleDrive-2arnavmana@gmail.com/My Drive'
PROJECT=DRIVE/'Projects'/'PhysioGraph'
PRIVATE=DRIVE/'Data'/'PhysioGraph_Biological_Discovery_20260905'
AMEND_ROOT=PROJECT/'research'/'preicu_linkage_amendment';AMEND_ROOT.mkdir(exist_ok=True)
OUT=AMEND_ROOT
PRIOR=PROJECT/'research'/'spo2_circulatory_context'
sys.path.insert(0,str(PROJECT/'src'))
from physiograph.etl.mimic_extractor import _mimic_lab_item_map
def amend_json(name,obj):
    (AMEND_ROOT/name).write_text(json.dumps(obj,indent=2,allow_nan=False,default=str)+'\n')
def amend_status(stage,status='running'):
    amend_json('run_status.json',dict(stage=stage,status=status,
        environment='google_colab' if IN_COLAB else 'local',
        updated_utc=datetime.now(timezone.utc).isoformat(),goal_complete=False))
    print(stage,status,flush=True)
lock=AMEND_ROOT/'protocol_lock.json'
if lock.exists():assert json.loads(lock.read_text())['sha256']==AMENDMENT_PROTOCOL_SHA256
else:amend_json('protocol_lock.json',dict(sha256=AMENDMENT_PROTOCOL_SHA256,
    locked_before_restored_values_extraction=True,original_feasibility_and_linkage_counts_seen=True,
    biological_interactions_and_directions_not_seen=True,utc=datetime.now(timezone.utc).isoformat()))
for folder,name,expected in [('spo2_hemoglobin','protocol_lock.json',HEMOGLOBIN_PROTOCOL_SHA256),
    ('spo2_troponin_direction','protocol_lock.json',DIRECTION_PROTOCOL_SHA256),
    ('spo2_troponin_direction','boundary_amendment.json',DIRECTION_BOUNDARY_SHA256)]:
    assert json.loads((PROJECT/'research'/folder/name).read_text())['sha256']==expected
validation=json.loads((PROJECT/'research/mimic_preicu_lab_linkage/known_id_validation.json').read_text())
for family in ['cbc_hemoglobin','troponin']:
    assert next(r for r in validation if r['family']==family)['passes_validation']
assert json.loads((PROJECT/'research/mimic_preicu_lab_linkage/independent_validation.json').read_text())['patient_time_assignment_matches_independent_pandas']
used=json.loads((PROJECT/'research/spo2_hemoglobin/source_manifest.json').read_text())['sources']
for key,info in used.items():
    s=Path(info['path']).stat()
    assert s.st_size==info['bytes'] and s.st_mtime_ns==info['mtime_ns'],key
checkpoints={}
for name in ['spo2_context','spo2_hemoglobin','troponin_direction','mimic_preicu_lab_linkage']:
    p=PRIVATE/(name+'.duckdb');s=p.stat();h=hashlib.sha256()
    with p.open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
    checkpoints[name]=dict(path=str(p),bytes=s.st_size,mtime_ns=s.st_mtime_ns,sha256=h.hexdigest())
amend_json('input_database_fingerprints.json',checkpoints)
def qs(value):return "'"+str(value).replace("'","''")+"'"
con=duckdb.connect(str(PRIVATE/'preicu_linkage_amendment.duckdb'))
con.execute('SET threads=2');con.execute("SET memory_limit='2GB'")
for alias,name in [('prior','spo2_context'),('hb','spo2_hemoglobin'),('ordering','troponin_direction'),('linkage','mimic_preicu_lab_linkage')]:
    con.execute(f"ATTACH {qs(PRIVATE/(name+'.duckdb'))} AS {alias} (READ_ONLY)")
amend_status('retrieving actual values for audited missing-ID laboratories')
lab=Path(used['MIMIC/labevents']['path'])
con.execute(f"""CREATE OR REPLACE TABLE mimic_restored_labs AS
 SELECT a.stay_id,try_cast(l.labevent_id AS BIGINT) labevent_id,
   try_cast(l.hadm_id AS BIGINT) source_hadm_id,a.candidate_hadm_id assigned_hadm_id,
   try_cast(l.itemid AS INTEGER) itemid,a.family,
   date_diff('second',c.admit_time,try_cast(l.charttime AS TIMESTAMP))/60.0 event_minute,
   date_diff('second',c.admit_time,try_cast(l.storetime AS TIMESTAMP))/60.0 available_minute,
   l.valueuom raw_unit,try_cast(l.valuenum AS DOUBLE) raw_value,'validated_temporal' linkage_class
 FROM read_csv({qs(lab)},header=true,all_varchar=true,sample_size=10000,strict_mode=true) l
 JOIN linkage.eligible_candidate_additions a ON try_cast(l.labevent_id AS BIGINT)=a.labevent_id
 JOIN prior.mimic_cohort c ON a.stay_id=c.stay_id AND a.candidate_hadm_id=c.hadm_id
   AND try_cast(l.subject_id AS BIGINT)=c.person_id
 WHERE a.family IN ('cbc_hemoglobin','troponin') AND coalesce(trim(l.hadm_id),'')=''""")
expected=con.execute("""SELECT count(*) FROM linkage.eligible_candidate_additions
 WHERE family IN ('cbc_hemoglobin','troponin')""").fetchone()[0]
assert con.execute('SELECT count(*) FROM mimic_restored_labs').fetchone()[0]==expected
assert con.execute('SELECT count(*)=count(DISTINCT labevent_id) FROM mimic_restored_labs').fetchone()[0]
assert con.execute("""SELECT count(*) FROM mimic_restored_labs
 WHERE source_hadm_id IS NOT NULL OR event_minute < -1440 OR event_minute>0""").fetchone()[0]==0
for key,info in used.items():
    s=Path(info['path']).stat()
    assert s.st_size==info['bytes'] and s.st_mtime_ns==info['mtime_ns'],key
for key,info in checkpoints.items():
    s=Path(info['path']).stat()
    assert s.st_size==info['bytes'] and s.st_mtime_ns==info['mtime_ns'],key
for d in ['eicu','mimic']:
    con.execute(f'CREATE OR REPLACE TABLE {d}_eligible AS SELECT * FROM hb.{d}_eligible')
    con.execute(f'CREATE OR REPLACE TABLE {d}_first_direction AS SELECT * FROM hb.{d}_first_direction')
    con.execute(f'CREATE OR REPLACE TABLE {d}_anchors AS SELECT * FROM ordering.{d}_anchors')
con.execute("""CREATE OR REPLACE TABLE eicu_hb_raw AS
 SELECT *, 'recorded_id' linkage_class FROM hb.eicu_hb_raw""")
con.execute("""CREATE OR REPLACE TABLE mimic_hb_raw AS
 SELECT *, 'recorded_id' linkage_class FROM hb.mimic_hb_raw
 UNION ALL SELECT stay_id,event_minute,available_minute,'Hemoglobin',3,raw_unit,'',raw_value,TRUE,linkage_class
 FROM mimic_restored_labs WHERE family='cbc_hemoglobin' AND itemid=51222""")
con.execute('CREATE OR REPLACE TABLE eicu_preicu_troponin AS SELECT * FROM ordering.eicu_preicu_troponin')
mapping=pd.DataFrame([dict(itemid=k,concept=v) for k,v in _mimic_lab_item_map(DRIVE/'Data/MIMIC/Full').items() if v in ['troponin_i','troponin_t']])
con.register('troponin_map',mapping)
con.execute("""CREATE OR REPLACE TABLE mimic_preicu_troponin AS
 SELECT stay_id,event_minute,available_minute,concept,unit,value FROM ordering.mimic_preicu_troponin
 UNION ALL SELECT r.stay_id,r.event_minute,r.available_minute,m.concept,lower(trim(coalesce(r.raw_unit,''))),r.raw_value
 FROM mimic_restored_labs r JOIN mimic_anchors a USING(stay_id) JOIN troponin_map m USING(itemid)
 WHERE r.family='troponin'""")
restored=con.execute("""SELECT family,count(*) records,count(DISTINCT stay_id) encounters
 FROM mimic_restored_labs GROUP BY family ORDER BY family""").df().to_dict('records')
amend_json('restored_record_counts.json',restored)
amend_json('source_manifest.json',dict(verified_fingerprints_reused_from=str(PRIOR/'input_manifest.json'),
    same_size_and_mtime_before_and_after=True,sources=used))
con.close()
print(json.dumps(restored,indent=2))
amend_status('restoration complete; unchanged feasibility algorithms pending')

retrieving actual values for audited missing-ID laboratories running
[
  {
    "family": "cbc_hemoglobin",
    "records": 2681,
    "encounters": 2506
  },
  {
    "family": "troponin",
    "records": 2009,
    "encounters": 1821
  }
]
restoration complete; unchanged feasibility algorithms pending running


In [18]:
# Reuse the exact original Hb normalization and support algorithm on augmented records.
OUT=AMEND_ROOT/'hemoglobin';OUT.mkdir(exist_ok=True)
def hb_json(name,obj):
    (OUT/name).write_text(json.dumps(obj,indent=2,allow_nan=False,default=str)+'\n')
def hb_status(stage,status='running'):
    hb_json('run_status.json',dict(stage=stage,status=status,
        environment='google_colab' if IN_COLAB else 'local',
        updated_utc=datetime.now(timezone.utc).isoformat(),goal_complete=False))
    print(stage,status,flush=True)
con=duckdb.connect(str(PRIVATE/'preicu_linkage_amendment.duckdb'))
con.execute('SET threads=2');con.execute("SET memory_limit='2GB'")
hb_status('hemoglobin normalization and endpoint availability')
coverage=[];quality=[];validation=[]
for d in ['eicu','mimic']:
    raw=con.execute(f'SELECT * FROM {d}_hb_raw').df()
    for key in ['unit_system','unit_interface']:
        raw[key+'_clean']=raw[key].fillna('').str.strip().str.lower().str.replace(' ','',regex=False)
    u=raw.unit_system_clean.where(raw.unit_system_clean.ne(''),raw.unit_interface_clean)
    conflict=raw.unit_system_clean.ne('')&raw.unit_interface_clean.ne('')&raw.unit_system_clean.ne(raw.unit_interface_clean)
    raw['unit_approved']=u.isin(['g/dl','g/l'])&~conflict
    raw['hemoglobin']=raw.raw_value/np.where(u.eq('g/l'),10.0,1.0)
    raw['quality_approved']=raw.source_approved.fillna(False)&raw.unit_approved&raw.hemoglobin.between(3,22)
    quality.append(dict(dataset=d,candidate_records=len(raw),
        wrong_source=int((~raw.source_approved.fillna(False)).sum()),
        unknown_or_conflicting_unit=int((~raw.unit_approved).sum()),
        outside_value_range_or_missing=int((~raw.hemoglobin.between(3,22)).sum()),
        accepted_records=int(raw.quality_approved.sum())))
    con.register('normalized_frame',raw)
    con.execute(f'CREATE OR REPLACE TABLE {d}_hb_normalized AS SELECT * FROM normalized_frame')
    con.execute(f"""CREATE OR REPLACE TABLE {d}_baseline_hb AS
      SELECT stay_id,event_minute,median(hemoglobin) hemoglobin,max(available_minute) available_minute
      FROM {d}_hb_normalized WHERE quality_approved GROUP BY stay_id,event_minute
      QUALIFY row_number() OVER(PARTITION BY stay_id ORDER BY event_minute DESC)=1""")
    independent=raw[raw.quality_approved].groupby(['stay_id','event_minute'],as_index=False).hemoglobin.median()
    independent=independent.sort_values('event_minute').drop_duplicates('stay_id',keep='last').set_index('stay_id')
    baseline=con.execute(f'SELECT * FROM {d}_baseline_hb').df().set_index('stay_id')
    assert set(independent.index)==set(baseline.index)
    np.testing.assert_allclose(independent.hemoglobin.sort_index(),baseline.hemoglobin.sort_index(),rtol=0,atol=1e-12)
    validation.append(dict(dataset=d,independent_latest_value_match=True,people=len(baseline)))
    support=con.execute(f"""SELECT e.exposed,count(*) eligible,count(b.hemoglobin) pre_icu_hemoglobin,
      sum(CASE WHEN b.hemoglobin IS NOT NULL AND e.troponin_observed THEN 1 ELSE 0 END) paired_endpoint_observed,
      sum(CASE WHEN b.hemoglobin IS NOT NULL AND b.available_minute>0 THEN 1 ELSE 0 END) result_entered_after_icu_admission
      FROM {d}_eligible e LEFT JOIN {d}_baseline_hb b USING(stay_id) GROUP BY e.exposed ORDER BY e.exposed""").df()
    observed={int(r.exposed):int(r.paired_endpoint_observed) for r in support.itertuples()}
    passes=sum(observed.values())>=200 and observed.get(0,0)>=50 and observed.get(1,0)>=50
    coverage.append(dict(dataset=d,passes_minimal_feasibility=passes,groups=support.to_dict('records')))
hb_json('coverage.json',coverage);hb_json('quality_audit.json',quality)
hb_json('independent_validation.json',validation)
hb_json('source_manifest.json',dict(verified_fingerprints_reused_from=str(PRIOR/'input_manifest.json'),
    same_size_and_mtime_before_and_after=True,sources=used))
hb_json('analysis_manifest.json',dict(protocol_sha256=HEMOGLOBIN_PROTOCOL_SHA256,linkage_amendment_sha256=AMENDMENT_PROTOCOL_SHA256,
    builder_sha256=hashlib.sha256((PROJECT/'scripts/build_biological_notebook.py').read_bytes()).hexdigest(),
    environment='google_colab' if IN_COLAB else 'local',completed_utc=datetime.now(timezone.utc).isoformat(),
    all_sources_pass=all(s['passes_minimal_feasibility'] for s in coverage),
    outcome_interactions_inspected=False,mortality_joined=False,synthetic_patients=0,biological_discovery=False))
con.close()
print(json.dumps(coverage,indent=2))
hb_status('hemoglobin susceptibility feasibility','complete')
amend_status('hemoglobin support complete; injury-ordering support pending')

hemoglobin normalization and endpoint availability running
[
  {
    "dataset": "eicu",
    "passes_minimal_feasibility": true,
    "groups": [
      {
        "exposed": 0,
        "eligible": 7143,
        "pre_icu_hemoglobin": 2445,
        "paired_endpoint_observed": 409.0,
        "result_entered_after_icu_admission": 151.0
      },
      {
        "exposed": 1,
        "eligible": 4309,
        "pre_icu_hemoglobin": 1539,
        "paired_endpoint_observed": 250.0,
        "result_entered_after_icu_admission": 69.0
      }
    ]
  },
  {
    "dataset": "mimic",
    "passes_minimal_feasibility": true,
    "groups": [
      {
        "exposed": 0,
        "eligible": 3313,
        "pre_icu_hemoglobin": 2387,
        "paired_endpoint_observed": 340.0,
        "result_entered_after_icu_admission": 91.0
      },
      {
        "exposed": 1,
        "eligible": 1398,
        "pre_icu_hemoglobin": 1063,
        "paired_endpoint_observed": 176.0,
        "result_entered_after_icu_admissi

In [19]:
# Reuse the exact original ordering-availability algorithm; do not compute trajectory directions.
OUT=AMEND_ROOT/'troponin_direction';OUT.mkdir(exist_ok=True)
def direction_json(name,obj):
    (OUT/name).write_text(json.dumps(obj,indent=2,allow_nan=False,default=str)+'\n')
def direction_status(stage,status='running'):
    direction_json('run_status.json',dict(stage=stage,status=status,
        environment='google_colab' if IN_COLAB else 'local',
        updated_utc=datetime.now(timezone.utc).isoformat(),goal_complete=False))
    print(stage,status,flush=True)
con=duckdb.connect(str(PRIVATE/'preicu_linkage_amendment.duckdb'))
con.execute('SET threads=2');con.execute("SET memory_limit='2GB'")
con.execute(f"ATTACH {qs(PRIVATE/'spo2_context.duckdb')} AS prior (READ_ONLY)")
direction_status('serial troponin trajectory support')
support=[]
for d in ['eicu','mimic']:
    con.execute(f"""CREATE OR REPLACE TABLE {d}_troponin AS
      WITH combined AS (
        SELECT stay_id,event_minute,available_minute,concept,unit,value FROM {d}_preicu_troponin
        UNION ALL
        SELECT l.stay_id,event_minute,available_minute,concept,unit,value FROM prior.{d}_context_labs l
        JOIN {d}_anchors a USING(stay_id) WHERE concept IN ('troponin_t','troponin_i'))
      SELECT stay_id,event_minute,concept,unit,median(value) AS value,max(available_minute) available_minute
      FROM combined WHERE value>0 AND value<=1000000 GROUP BY stay_id,event_minute,concept,unit""")
    con.execute(f"""CREATE OR REPLACE TABLE {d}_trajectory_support AS
      WITH pre_assays AS (
        SELECT t.stay_id,t.concept,t.unit,count(*) pre_n,min(event_minute) pre_first_time,
        max(event_minute) pre_last_time FROM {d}_troponin t JOIN {d}_anchors a USING(stay_id)
        WHERE event_minute>=a.episode_minute-720 AND event_minute<a.episode_minute
          AND event_minute>=a.hospital_start_minute
        GROUP BY t.stay_id,t.concept,t.unit), chosen AS (
        SELECT * FROM pre_assays QUALIFY row_number() OVER(PARTITION BY stay_id
          ORDER BY pre_first_time,CASE WHEN concept='troponin_t' THEN 0 ELSE 1 END,unit)=1), post_assays AS (
        SELECT c.stay_id,count(t.event_minute) post_n FROM chosen c JOIN {d}_anchors a USING(stay_id)
        LEFT JOIN {d}_troponin t ON t.stay_id=c.stay_id AND t.concept=c.concept AND t.unit=c.unit
          AND t.event_minute>=a.episode_minute+120 AND t.event_minute<=a.episode_minute+720
        GROUP BY c.stay_id)
      SELECT a.stay_id,a.episode_minute,c.concept,c.unit,c.pre_n,c.pre_first_time,c.pre_last_time,
        coalesce(p.post_n,0) post_n,
        coalesce(c.pre_n>=2 AND c.pre_last_time-c.pre_first_time>=60,FALSE) serial_pre,
        coalesce(c.pre_n>=2 AND c.pre_last_time-c.pre_first_time>=60 AND p.post_n>=1,FALSE) complete_trajectory
      FROM {d}_anchors a LEFT JOIN chosen c USING(stay_id) LEFT JOIN post_assays p USING(stay_id)""")
    s=con.execute(f'SELECT * FROM {d}_trajectory_support').df()
    support.append(dict(dataset=d,episodes_with_hospital_boundary=len(s),
        any_pre_episode_assay=int(s.pre_n.notna().sum()),serial_pre_episode=int(s.serial_pre.sum()),
        complete_pre_post_trajectories=int(s.complete_trajectory.sum()),
        passes_count_gate=bool(s.complete_trajectory.sum()>=100)))
direction_json('trajectory_support.json',support)
direction_json('analysis_manifest.json',dict(environment='google_colab' if IN_COLAB else 'local',
    protocol_sha256=DIRECTION_PROTOCOL_SHA256,linkage_amendment_sha256=AMENDMENT_PROTOCOL_SHA256,boundary_amendment_sha256=DIRECTION_BOUNDARY_SHA256,
    completed_utc=datetime.now(timezone.utc).isoformat(),
    all_sources_pass=all(s['passes_count_gate'] for s in support),
    trajectory_directions_inspected=False,mortality_joined=False,synthetic_patients=0,
    biological_discovery=False))
con.close()
print(pd.DataFrame(support).to_string(index=False))
direction_status('biological-ordering feasibility','complete')
for subdir,original,name in [('hemoglobin','spo2_hemoglobin','coverage.json'),('troponin_direction','spo2_troponin_direction','trajectory_support.json')]:
    before=json.loads((PROJECT/'research'/original/name).read_text())
    after=json.loads((AMEND_ROOT/subdir/name).read_text())
    assert next(r for r in before if r['dataset']=='eicu')==next(r for r in after if r['dataset']=='eicu')
amend_json('analysis_manifest.json',dict(amendment_sha256=AMENDMENT_PROTOCOL_SHA256,
    original_hemoglobin_protocol_sha256=HEMOGLOBIN_PROTOCOL_SHA256,
    original_direction_protocol_sha256=DIRECTION_PROTOCOL_SHA256,
    original_direction_boundary_sha256=DIRECTION_BOUNDARY_SHA256,
    environment='google_colab' if IN_COLAB else 'local',completed_utc=datetime.now(timezone.utc).isoformat(),
    hemoglobin_feasibility_pass=json.loads((AMEND_ROOT/'hemoglobin/analysis_manifest.json').read_text())['all_sources_pass'],
    ordering_feasibility_pass=json.loads((AMEND_ROOT/'troponin_direction/analysis_manifest.json').read_text())['all_sources_pass'],
    biological_associations_examined=False,troponin_directions_examined=False,biological_discovery=False))
amend_status('linkage-only amended feasibility','complete')

serial troponin trajectory support running
dataset  episodes_with_hospital_boundary  any_pre_episode_assay  serial_pre_episode  complete_pre_post_trajectories  passes_count_gate
   eicu                             4309                   1326                 224                             133               True
  mimic                             1398                    608                 132                              76              False
biological-ordering feasibility complete
linkage-only amended feasibility complete


# Hemoglobin modification of the existing SpO2 association

Frozen before hemoglobin-by-outcome inspection. Only MIMIC and eICU; both
databases have been explored previously. This is an observational biological
susceptibility test, not an estimate of transfusion benefit or causal oxygen
delivery. The broad anemia–hypoxemia hypothesis is already established in
prior literature; exact statistical novelty would not establish a mechanism.

## Population, exposure and endpoints

Use the unchanged dynamics-eligible HF risk set and latest pre-ICU CBC Hb
selected under the original feasibility protocol plus the documented,
validated missing-hospital-ID linkage amendment. Preserve the explicit-ID-only
selection for sensitivity. Never change source windows or endpoint definitions.
X is the existing first-four-hour absolute SpO2 jump indicator, including rises.
H=(12−Hb[g/dL])/2 is continuous, with no optimized threshold or spline search.
The two primary endpoints are the existing assay-matched 12-hour troponin rise
indicator and hospital mortality, fitted separately in each database. Missing
Hb and outcomes are not imputed. The primary population is patients with
observed pre-ICU Hb, not all HF patients. Troponin is a myocardial-injury proxy,
not adjudicated infarction; discharge mortality is not a fixed-time endpoint.

## Frozen models and scale

Fit source-specific binomial-logit GEE with independence working correlation
and patient-clustered robust covariance. Include intercept, X, H and X*H.
The primary interaction estimand is exp(beta[X*H]): the ratio of the SpO2
exposure odds ratios for each 2 g/dL lower Hb. Do not call it a risk ratio.
Use two-sided Wald inference and Holm correction over exactly four primary
interaction tests (two endpoints by two databases). Non-estimable tests occupy
their family slot with p=1. No pooled rescue of discordant results.

Covariates, all fixed: age/10, recorded male sex and unknown-sex flag;
(mean SpO2−95)/5; proportion of observed SpO2 bins below 90%; log(1+number of
source SpO2 readings); mean HR/20; mean MAP/20; log(creatinine), log(lactate),
maximum documented FiO2/20; documented invasive ventilation and documented
vasoactive/inotropic infusion in ICU minutes [0,240). Also include H times
centered mean SpO2 and H times hypoxemic-bin proportion to distinguish a
dynamics interaction from modification of absolute oxygenation. Do not
substitute discharge diagnoses as prospectively measured illness severity.

For continuous missing covariates use source-and-analysis-population median
plus a missingness indicator, including age if necessary. All-missing columns
are assigned zero; omit only constant columns (other than the intercept),
never X, H, or X*H. No collinearity-driven variable selection; rank deficiency
or invalid covariance makes a model non-estimable. Primary model requires
100 events, 100 non-events, 100 distinct persons and Hb SD >=0.5 g/dL;
report counts, full design dimension, events per parameter, convergence,
rank, condition number, robust SE and uncertainty. These are minimal
estimability checks, not a claim of adequate interaction power. A wide CI
that includes important amplification is inconclusive, not evidence of none.

At Hb=8 and 12 g/dL, standardize fitted risks over the same observed covariate
distribution at X=0 and X=1, updating all H interactions. Report risk
differences and their difference, with delta-method robust 95% CIs. Compute
these only if each of the four actual Hb neighborhoods (8±1 and 12±1, by X)
contains >=20 patients. These conditional model predictions are associative;
setting Hb in a regression is not an intervention on anemia or transfusion.

## Treatment and physiology source rules

Use existing production-quality raw SpO2 bins and unchanged first-four-hour
MAP, HR, creatinine and lactate definitions. Existing ICU laboratory covariates
have substantial missingness; no unvalidated new admission linkage for them.
FiO2 is a documented inspired oxygen fraction, converted from 0.20–1 to percent
and retained only at 21–100%. MIMIC dictionary labels must match inspired O2
fraction or FiO2, excluding ECMO/TandemHeart contexts and warning=1 records.
eICU respiratory labels must match FiO2/fraction inspired oxygen/inspired O2/
oxygen concentration. Preserve and audit raw labels and invalid-value counts.

MIMIC invasive ventilation requires procedure item 225792 overlapping [0,240).
eICU requires a literal pipe-delimited mechanical ventilation treatment
segment, excluding noninvasive ventilation and ventilator weaning. Generic
respiratory modes alone cannot establish an invasive airway.
MIMIC vasoactive drugs: 221906,221289,221662,221653,221749,222315,221986,
positive rate, overlapping [0,240), excluding Rewritten records. eICU uses
positive drugrate (or infusionrate if drugrate is missing) with exact drug/
brand word matches for norepinephrine/Levophed, epinephrine/adrenaline,
dopamine, dobutamine/Dobutrex, phenylephrine/Neo-Synephrine, vasopressin,
milrinone/Primacor at infusionoffset in [0,240). No dose-equivalence inference.
These are documentation indicators; absence of documentation is not confirmed
absence of treatment. eICU infusionoffset is entry timing. Context overlaps
the exposure window, so adjustment can include responses to instability.

## Sensitivity and interpretation

Repeat the fixed model using explicit-ID-only Hb selection. Repeat without
care-context covariates to reveal adjustment sensitivity, without promoting
that reduced model to primary. Report Hb measurement/store/revision timing;
exclude records entered after ICU admission as a separate sensitivity, while
acknowledging that eICU revision time is not necessarily first availability.
Compare first-drop and first-rise interactions descriptively if primary
replication warrants follow-up; recovery rises are not a negative control.

For troponin ascertainment, prespecify a sensitivity observation model using
the same design among all Hb-observed patients: five folds grouped by person,
L2 logistic regression C=1, training-fold median imputation and scaling,
random seed 20260905. Use inverse probability weights with probability floor
0.02 and report fraction below the floor, calibration, and Kish ESS overall
and by X. Weighted inference is not credible if >10% target probabilities
are below 0.02 or either exposure group has ESS<50. Missing-at-random remains
untestable and weighting does not repair unavailable confounders.

Advance this candidate only if both databases have positive primary
interactions with Holm p<0.05 for the same endpoint, directionally consistent
sensitivities, and evidence of a substantial association (interaction odds
ratio >=1.5 per 2 g/dL lower Hb, or >=5 percentage-point mortality difference
in differences). This is a research-screen threshold, not proof of clinical
utility. A positive result requires further specification checks and a
prospective validation/intervention strategy. Residual confounding by bleeding,
transfusion, illness severity, oxygen treatment and pulse-oximetry error
precludes a mortality-reduction claim. A negative or imprecise result is
reported unchanged; no thresholds/endpoints are tuned to obtain significance.

All code is embedded in the single primary notebook. Patient-level tables
remain in private Drive Data; publish aggregate outputs only. Record protocol
hash before association computation, source provenance and actual execution
environment. No result from this experiment by itself completes the user's
paradigm-shifting biological-discovery goal.

In [20]:
HB_INTERACTION_PROTOCOL_SHA256='81e747927dc21a358619fce26f541680d75c41ebd2a8c5993101ab6d5f5388ac'

import sys,json,hashlib,importlib.util,subprocess
from pathlib import Path
from datetime import datetime,timezone
import numpy as np
import pandas as pd
import duckdb
IN_COLAB=importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
DRIVE=Path('/content/drive/MyDrive') if IN_COLAB else Path.home()/'Library/CloudStorage/GoogleDrive-2arnavmana@gmail.com/My Drive'
PROJECT=DRIVE/'Projects/PhysioGraph'
PRIVATE=DRIVE/'Data/PhysioGraph_Biological_Discovery_20260905'
OUT=PROJECT/'research/spo2_hemoglobin_interaction';OUT.mkdir(exist_ok=True)
def hj(name,obj):
    (OUT/name).write_text(json.dumps(obj,indent=2,allow_nan=False,default=str)+'\n')
def hs(stage,status='running'):
    hj('run_status.json',dict(stage=stage,status=status,utc=datetime.now(timezone.utc).isoformat(),
        environment='google_colab' if IN_COLAB else 'local',goal_complete=False))
    print(stage,status,flush=True)
lock=OUT/'protocol_lock.json'
if lock.exists():assert json.loads(lock.read_text())['sha256']==HB_INTERACTION_PROTOCOL_SHA256
else:hj('protocol_lock.json',dict(sha256=HB_INTERACTION_PROTOCOL_SHA256,
    utc=datetime.now(timezone.utc).isoformat(),hemoglobin_outcome_associations_not_examined=True,
    environment='google_colab' if IN_COLAB else 'local'))
def qs(v):return "'"+str(v).replace("'","''")+"'"
hs('care-context extraction setup')

care-context extraction setup running


In [21]:
# Extract treatment and oxygen context without joining Hb concentrations or outcomes.
con=duckdb.connect(str(PRIVATE/'hemoglobin_interaction.duckdb'))
con.execute('SET threads=2');con.execute("SET memory_limit='2GB'")
con.execute(f"ATTACH {qs(PRIVATE/'spo2_context.duckdb')} AS prior (READ_ONLY)")
previous=json.loads((PROJECT/'research/spo2_circulatory_context/input_manifest.json').read_text())
used={}
def source(dataset,name):
    key=dataset+'/'+name;p=DRIVE/'Data'/dataset/'Full'/(name+'.csv');s=p.stat()
    old=previous.get(key)
    if old:
        assert s.st_size==old['bytes'] and s.st_mtime_ns==old['mtime_ns'],key
        info={**old,'fingerprint_reused_from':'spo2_circulatory_context/input_manifest.json'}
    else:
        h=hashlib.sha256()
        with p.open('rb') as f:
            for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
        info=dict(path=str(p),bytes=s.st_size,mtime_ns=s.st_mtime_ns,sha256=h.hexdigest())
    used[key]=info
    return f'read_csv({qs(p)},header=true,all_varchar=true,sample_size=10000,strict_mode=true,delim={qs(",")},quote={qs(chr(34))},escape={qs(chr(34))})'
for d in ['eicu','mimic']:
    con.execute(f'''CREATE OR REPLACE TABLE {d}_context_eligible AS
      SELECT c.stay_id,c.admit_time FROM prior.{d}_cohort c
      JOIN prior.{d}_features f USING(stay_id)
      WHERE f.spo2_bins>=3 AND f.spo2_transitions>=2''')
hs('eICU respiratory oxygen extraction')
resp=source('eICU','respiratoryCharting')
con.execute(rf'''CREATE OR REPLACE TABLE eicu_fio2_raw AS
  SELECT e.stay_id,try_cast(r.respchartoffset AS DOUBLE) event_minute,
    concat_ws(' ',r.respcharttypecat,r.respchartvaluelabel) raw_label,
    r.respchartvalue raw_value,
    try_cast(regexp_extract(r.respchartvalue,'([-+]?[0-9]*[.]?[0-9]+)',1) AS DOUBLE) numeric_value
  FROM {resp} r JOIN eicu_context_eligible e ON try_cast(r.patientunitstayid AS BIGINT)=e.stay_id
  WHERE try_cast(r.respchartoffset AS DOUBLE)>=0 AND try_cast(r.respchartoffset AS DOUBLE)<240
    AND regexp_matches(lower(concat_ws(' ',r.respcharttypecat,r.respchartvaluelabel)),
        '\bfio2\b|fraction inspired oxygen|inspired o2|oxygen concentration')''')
hs('eICU documented circulatory support extraction')
inf=source('eICU','infusionDrug')
con.execute(rf'''CREATE OR REPLACE TABLE eicu_pressor_raw AS
  SELECT e.stay_id,l.drugname raw_label,try_cast(l.infusionoffset AS DOUBLE) event_minute,
    coalesce(try_cast(l.drugrate AS DOUBLE),try_cast(l.infusionrate AS DOUBLE)) positive_rate
  FROM {inf} l JOIN eicu_context_eligible e ON try_cast(l.patientunitstayid AS BIGINT)=e.stay_id
  WHERE try_cast(l.infusionoffset AS DOUBLE)>=0 AND try_cast(l.infusionoffset AS DOUBLE)<240
  AND regexp_matches(lower(l.drugname),
    '\b(norepinephrine|levophed|epinephrine|adrenaline|dopamine|dobutamine|dobutrex|phenylephrine|neo[ -]?synephrine|vasopressin|milrinone|primacor)\b')''')
hs('eICU invasive ventilation documentation extraction')
treat=source('eICU','treatment')
con.execute(rf'''CREATE OR REPLACE TABLE eicu_vent_raw AS
  SELECT e.stay_id,l.treatmentstring raw_label,try_cast(l.treatmentoffset AS DOUBLE) event_minute,
    regexp_matches(lower(l.treatmentstring),'(^|\|)\s*mechanical ventilation\s*(\||$)')
      AND NOT regexp_matches(lower(l.treatmentstring),'non[ -]?invasive ventilation|ventilator weaning') accepted
  FROM {treat} l JOIN eicu_context_eligible e ON try_cast(l.patientunitstayid AS BIGINT)=e.stay_id
  WHERE try_cast(l.treatmentoffset AS DOUBLE)>=0 AND try_cast(l.treatmentoffset AS DOUBLE)<240
    AND regexp_matches(lower(l.treatmentstring),'ventilat')''')
hs('MIMIC vasoactive infusion extraction')
inf=source('MIMIC','inputevents')
con.execute(f'''CREATE OR REPLACE TABLE mimic_pressor_raw AS
  SELECT e.stay_id,l.itemid raw_label,l.statusdescription,
    date_diff('second',e.admit_time,try_cast(l.starttime AS TIMESTAMP))/60.0 event_minute,
    date_diff('second',e.admit_time,try_cast(l.endtime AS TIMESTAMP))/60.0 end_minute,
    try_cast(l.rate AS DOUBLE) positive_rate
  FROM {inf} l JOIN mimic_context_eligible e ON try_cast(l.stay_id AS BIGINT)=e.stay_id
  WHERE try_cast(l.itemid AS BIGINT) IN (221906,221289,221662,221653,221749,222315,221986)
    AND try_cast(l.starttime AS TIMESTAMP)<e.admit_time+INTERVAL '240 minutes'
    AND try_cast(l.endtime AS TIMESTAMP)>e.admit_time
    AND try_cast(l.endtime AS TIMESTAMP)>=try_cast(l.starttime AS TIMESTAMP)
    AND lower(coalesce(l.statusdescription,''))<>'rewritten' ''')
hs('MIMIC invasive ventilation extraction')
proc=source('MIMIC','procedureevents')
con.execute(f'''CREATE OR REPLACE TABLE mimic_vent_raw AS
  SELECT e.stay_id,l.itemid raw_label,
    date_diff('second',e.admit_time,try_cast(l.starttime AS TIMESTAMP))/60.0 event_minute,
    date_diff('second',e.admit_time,try_cast(l.endtime AS TIMESTAMP))/60.0 end_minute,true accepted
  FROM {proc} l JOIN mimic_context_eligible e ON try_cast(l.stay_id AS BIGINT)=e.stay_id
  WHERE try_cast(l.itemid AS BIGINT)=225792
    AND try_cast(l.starttime AS TIMESTAMP)<e.admit_time+INTERVAL '240 minutes'
    AND try_cast(l.endtime AS TIMESTAMP)>e.admit_time
    AND try_cast(l.endtime AS TIMESTAMP)>=try_cast(l.starttime AS TIMESTAMP)''')
hs('MIMIC FiO2 dictionary and chart extraction')
items=source('MIMIC','d_items')
con.execute(rf'''CREATE OR REPLACE TABLE mimic_fio2_items AS
  SELECT try_cast(itemid AS BIGINT) itemid,label,category FROM {items}
  WHERE lower(linksto)='chartevents'
    AND regexp_matches(lower(label),'inspired o2 fraction|\bfio2\b')
    AND NOT regexp_matches(lower(coalesce(category,'')||' '||label),'ecmo|tandemheart|\(ch\)')''')
hj('mimic_fio2_dictionary.json',con.execute('SELECT * FROM mimic_fio2_items').df().to_dict('records'))
charts=source('MIMIC','chartevents')
con.execute(f'''CREATE OR REPLACE TABLE mimic_fio2_raw AS
  SELECT e.stay_id,date_diff('second',e.admit_time,try_cast(l.charttime AS TIMESTAMP))/60.0 event_minute,
    m.label raw_label,l.value raw_value,try_cast(l.valuenum AS DOUBLE) numeric_value
  FROM {charts} l JOIN mimic_context_eligible e ON try_cast(l.stay_id AS BIGINT)=e.stay_id
  JOIN mimic_fio2_items m ON try_cast(l.itemid AS BIGINT)=m.itemid
  WHERE try_cast(l.charttime AS TIMESTAMP)>=e.admit_time
    AND try_cast(l.charttime AS TIMESTAMP)<e.admit_time+INTERVAL '240 minutes'
    AND coalesce(try_cast(l.warning AS INTEGER),0)<>1''')
audit={};coverage=[]
for d in ['eicu','mimic']:
    con.execute(f'''CREATE OR REPLACE TABLE {d}_fio2 AS
      SELECT *,CASE WHEN numeric_value BETWEEN 0.20 AND 1 THEN numeric_value*100
        ELSE numeric_value END fio2_percent FROM {d}_fio2_raw''')
    audit[d+'_fio2']=con.execute(f'''SELECT raw_label,count(*) records,
      count(*) FILTER(WHERE fio2_percent BETWEEN 21 AND 100) accepted,
      min(numeric_value) numeric_min,max(numeric_value) numeric_max
      FROM {d}_fio2 GROUP BY raw_label ORDER BY records DESC''').df().replace({np.nan:None}).to_dict('records')
    audit[d+'_pressor']=con.execute(f'''SELECT raw_label,count(*) records,
      count(*) FILTER(WHERE positive_rate>0) positive_records FROM {d}_pressor_raw
      GROUP BY raw_label ORDER BY records DESC''').df().to_dict('records')
    audit[d+'_vent']=con.execute(f'''SELECT raw_label,count(*) records,
      count(*) FILTER(WHERE accepted) accepted FROM {d}_vent_raw
      GROUP BY raw_label ORDER BY records DESC''').df().to_dict('records')
    con.execute(f'''CREATE OR REPLACE TABLE {d}_care AS
      SELECT e.stay_id,f.fio2_max,coalesce(v.vent_documented,0) vent_documented,
        coalesce(p.vasoactive_documented,0) vasoactive_documented
      FROM {d}_context_eligible e
      LEFT JOIN (SELECT stay_id,max(fio2_percent) fio2_max FROM {d}_fio2
        WHERE fio2_percent BETWEEN 21 AND 100 GROUP BY stay_id) f USING(stay_id)
      LEFT JOIN (SELECT stay_id,1 vent_documented FROM {d}_vent_raw WHERE accepted GROUP BY stay_id) v USING(stay_id)
      LEFT JOIN (SELECT stay_id,1 vasoactive_documented FROM {d}_pressor_raw WHERE positive_rate>0 GROUP BY stay_id) p USING(stay_id)''')
    coverage.append(dict(dataset=d,**con.execute(f'''SELECT count(*) eligible,count(fio2_max) fio2_observed,
      sum(vent_documented) documented_invasive_ventilation,sum(vasoactive_documented) documented_vasoactive
      FROM {d}_care''').df().iloc[0].to_dict()))
for key,info in used.items():
    s=Path(info['path']).stat();assert s.st_size==info['bytes'] and s.st_mtime_ns==info['mtime_ns'],key
hj('source_manifest.json',used);hj('care_label_audit.json',audit);hj('care_coverage.json',coverage)
con.close()
print(json.dumps(coverage,indent=2,default=str))
hs('care-context extraction','complete')

eICU respiratory oxygen extraction running
eICU documented circulatory support extraction running
eICU invasive ventilation documentation extraction running
MIMIC vasoactive infusion extraction running
MIMIC invasive ventilation extraction running
MIMIC FiO2 dictionary and chart extraction running
[
  {
    "dataset": "eicu",
    "eligible": 11452.0,
    "fio2_observed": 4224.0,
    "documented_invasive_ventilation": 1596.0,
    "documented_vasoactive": 956.0
  },
  {
    "dataset": "mimic",
    "eligible": 4711.0,
    "fio2_observed": 1883.0,
    "documented_invasive_ventilation": 1450.0,
    "documented_vasoactive": 1526.0
  }
]
care-context extraction complete


In [22]:
# Open Hb-by-outcome associations only under the already frozen specification.
import warnings
import statsmodels.api as sm
from scipy.special import expit
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests
if importlib.util.find_spec('sklearn') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','scikit-learn==1.6.1'])
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss,roc_auc_score
hs('frozen hemoglobin interaction analysis')
con=duckdb.connect(str(PRIVATE/'hemoglobin_interaction.duckdb'))
import re
care_checks=[]
for d in ['eicu','mimic']:
    f=con.execute(f'SELECT * FROM {d}_fio2_raw').df()
    if d=='mimic':assert set(f.raw_label)=={'Inspired O2 Fraction'}
    else:assert set(f.raw_label)<= {'respFlowSettings FiO2','respFlowCareData FIO2 (%)',
        'respFlowCareData FiO2','respFlowCareData Set Fraction of Inspired Oxygen (FIO2)'}
    f['normalized']=f.numeric_value.where(~f.numeric_value.between(.2,1),f.numeric_value*100)
    maxima=f.loc[f.normalized.between(21,100)].groupby('stay_id').normalized.max().sort_index()
    care=con.execute(f'SELECT * FROM {d}_care ORDER BY stay_id').df().set_index('stay_id')
    assert np.allclose(care.fio2_max.dropna(),maxima)
    assert care.fio2_max.dropna().index.equals(maxima.index)
    v=con.execute(f'SELECT * FROM {d}_vent_raw').df()
    if d=='eicu':
        accepted=v.raw_label.map(lambda s:'mechanical ventilation' in [x.strip() for x in s.lower().split('|')]
            and not re.search(r'non[ -]?invasive ventilation|ventilator weaning',s.lower()))
        assert (accepted.astype(bool)==v.accepted).all()
    else:accepted=v.accepted
    expected_vent=set(v.loc[accepted,'stay_id'])
    assert expected_vent==set(care.index[care.vent_documented==1])
    p=con.execute(f'SELECT * FROM {d}_pressor_raw').df()
    assert set(p.loc[p.positive_rate>0,'stay_id'])==set(care.index[care.vasoactive_documented==1])
    care_checks.append(dict(dataset=d,independent_normalization_and_aggregation_match=True,
        fio2_challenge_fields_contribute_no_rows=True if d=='mimic' else None,
        invasive_ventilation_patient_set_matches=True,positive_rate_patient_set_matches=True))
hj('independent_care_validation.json',care_checks)
for alias,name in [('prior','spo2_context'),('amend','preicu_linkage_amendment'),('original','spo2_hemoglobin')]:
    con.execute(f"ATTACH {qs(PRIVATE/(name+'.duckdb'))} AS {alias} (READ_ONLY)")
fingerprints={}
for name in ['spo2_context','preicu_linkage_amendment','spo2_hemoglobin']:
    p=PRIVATE/(name+'.duckdb');s=p.stat();h=hashlib.sha256()
    with p.open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
    fingerprints[name]=dict(path=str(p),sha256=h.hexdigest(),bytes=s.st_size,mtime_ns=s.st_mtime_ns)
hj('model_input_fingerprints.json',fingerprints)
frames={};availability=[]
for d in ['eicu','mimic']:
    con.execute(f'''CREATE OR REPLACE TABLE {d}_analysis AS
      SELECT e.stay_id,e.exposed,c.person_id,c.age,c.is_male,c.sex_unknown_flag,
        c.hospital_mortality,c.troponin_rise_12h,c.baseline_creatinine,c.baseline_lactate,
        h.hemoglobin,h.event_minute hb_event_minute,h.available_minute hb_available_minute,
        o.hemoglobin original_hemoglobin,o.available_minute original_hb_available_minute,
        f.spo2_mean,f.hr_mean,f.map_mean,s.hypoxic_fraction,s.spo2_readings,
        k.fio2_max,k.vent_documented,k.vasoactive_documented
      FROM amend.{d}_eligible e JOIN prior.{d}_context_endpoints c USING(stay_id)
      JOIN amend.{d}_baseline_hb h USING(stay_id)
      LEFT JOIN original.{d}_baseline_hb o USING(stay_id)
      JOIN prior.{d}_features f USING(stay_id) JOIN {d}_care k USING(stay_id)
      JOIN (SELECT stay_id,avg(CASE WHEN value<90 THEN 1.0 ELSE 0.0 END) hypoxic_fraction,
        sum(n_readings) spo2_readings FROM prior.{d}_bins WHERE concept='spo2' GROUP BY stay_id) s USING(stay_id)''')
    df=con.execute(f'SELECT * FROM {d}_analysis ORDER BY stay_id').df()
    assert df.stay_id.is_unique and df.person_id.notna().all()
    assert len(df)==({'mimic':3450,'eicu':3984}[d])
    assert int(df.troponin_rise_12h.notna().sum())==({'mimic':516,'eicu':659}[d])
    assert df.exposed.isin([0,1]).all()
    frames[d]=df
    availability.append(dict(dataset=d,n=len(df),unique_persons=df.person_id.nunique(),
        hb_entered_after_icu=int((df.hb_available_minute>0).sum()),
        missing={k:int(df[k].isna().sum()) for k in ['age','spo2_mean','hypoxic_fraction','hr_mean','map_mean',
          'baseline_creatinine','baseline_lactate','fio2_max','troponin_rise_12h','hospital_mortality']}))
hj('analysis_availability.json',availability)

def design(df,care=True,medians=None):
    z=pd.DataFrame(index=df.index)
    z['intercept']=1.0;z['X']=df.exposed.astype(float);z['H']=(12-df.hemoglobin)/2
    z['X_H']=z.X*z.H
    vals=dict(age=df.age/10,mean_spo2=(df.spo2_mean-95)/5,
        hypoxic_fraction=df.hypoxic_fraction,log_readings=np.log1p(df.spo2_readings),
        mean_hr=df.hr_mean/20,mean_map=df.map_mean/20,
        log_creatinine=np.log(df.baseline_creatinine.where(df.baseline_creatinine>0)),
        log_lactate=np.log(df.baseline_lactate.where(df.baseline_lactate>0)))
    if care:vals['fio2']=df.fio2_max/20
    fit_medians={} if medians is None else dict(medians)
    for key,v in vals.items():
        v=v.replace([np.inf,-np.inf],np.nan)
        if medians is None:fit_medians[key]=float(v.median()) if v.notna().any() else 0.0
        z[key]=v.fillna(fit_medians[key]);z[key+'_missing']=v.isna().astype(float)
    z['male']=df.is_male.fillna(0).astype(float)
    z['sex_unknown']=df.sex_unknown_flag.fillna(1).astype(float)
    if care:
        z['vent_documented']=df.vent_documented.astype(float)
        z['vasoactive_documented']=df.vasoactive_documented.astype(float)
    z['H_mean_spo2']=z.H*z.mean_spo2
    z['H_hypoxic_fraction']=z.H*z.hypoxic_fraction
    assert np.isfinite(z.to_numpy()).all()
    return z,fit_medians

def standardized(df,z,fit):
    neighborhoods={f'hb{hb}_X{x}':int(((df.hemoglobin.between(hb-1,hb+1))&(df.exposed==x)).sum())
        for hb in [8,12] for x in [0,1]}
    result=dict(neighborhoods=neighborhoods,support_pass=min(neighborhoods.values())>=20)
    if not result['support_pass']:return result
    estimates={};grads={};cov=np.asarray(fit.cov_params())
    for hb in [8,12]:
        for x in [0,1]:
            a=z.copy();a['X']=x;a['H']=(12-hb)/2;a['X_H']=a.X*a.H
            for name,parent in [('H_mean_spo2','mean_spo2'),('H_hypoxic_fraction','hypoxic_fraction')]:
                if name in a:a[name]=a.H*a[parent]
            av=a.to_numpy();p=expit(av@np.asarray(fit.params));g=(av*(p*(1-p))[:,None]).mean(axis=0)
            key=f'hb{hb}_X{x}';v=float(p.mean());se=float(np.sqrt(max(0,g@cov@g)))
            estimates[key]=dict(risk=v,ci95=[v-1.96*se,v+1.96*se]);grads[key]=g
    contrasts={}
    for hb in [8,12]:
        a,b=f'hb{hb}_X1',f'hb{hb}_X0';v=estimates[a]['risk']-estimates[b]['risk'];g=grads[a]-grads[b]
        se=float(np.sqrt(max(0,g@cov@g)));contrasts[f'RD_hb{hb}']=dict(value=v,ci95=[v-1.96*se,v+1.96*se])
    v=contrasts['RD_hb8']['value']-contrasts['RD_hb12']['value']
    g=grads['hb8_X1']-grads['hb8_X0']-grads['hb12_X1']+grads['hb12_X0']
    se=float(np.sqrt(max(0,g@cov@g)))
    result.update(risks=estimates,contrasts=contrasts,difference_in_differences=dict(value=v,ci95=[v-1.96*se,v+1.96*se]))
    return result

fits={};results=[];design_information=[]
def fit_one(d,df,endpoint,variant,care=True,weights=None):
    a=df.loc[df[endpoint].notna()].copy()
    y=a[endpoint].astype(float);assert y.isin([0,1]).all()
    z,med=design(a,care=care)
    constants=[k for k in z if k!='intercept' and z[k].nunique()==1]
    invalid_required=bool(set(constants)&{'X','H','X_H'})
    z=z.drop(columns=[k for k in constants if k not in ['X','H','X_H']])
    rank=int(np.linalg.matrix_rank(z));n=len(a);events=int(y.sum());clusters=int(a.person_id.nunique())
    info=dict(dataset=d,endpoint=endpoint,variant=variant,n=n,events=events,non_events=n-events,
        persons=clusters,parameters=z.shape[1],events_per_parameter=events/z.shape[1],
        hb_sd=float(a.hemoglobin.std()),columns=list(z),constant_columns_omitted=constants,
        rank=rank,condition_number=float(np.linalg.cond(z)),medians=med,estimable=False)
    if min(events,n-events)<100 or clusters<100 or a.hemoglobin.std()<0.5 or invalid_required or rank<z.shape[1]:
        info['reason']='Prespecified information, exposure variation or full-rank gate failed'
        results.append(info);return
    if variant=='primary':
        # Planning approximation using a hypothetical constant risk; no observed y enters this calculation.
        inv=np.linalg.inv(z.to_numpy().T@z.to_numpy());j=list(z).index('X_H')
        planning=[]
        for risk in [.1,.3,.5]:
            se=float(np.sqrt(inv[j,j]/(risk*(1-risk))))
            planning.append(dict(assumed_constant_outcome_risk=risk,independent_fisher_se=se,
                interaction_OR_for_80_percent_power_bonferroni4=float(np.exp((norm.ppf(1-.05/8)+norm.ppf(.8))*se))))
        design_information.append(dict(dataset=d,endpoint=endpoint,planning=planning,
            limitation='Independent-observation null-risk approximation, not empirical clustered power'))
        hj('design_information.json',design_information)
    w=None if weights is None else np.asarray(weights.loc[a.index],float)
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            fit=sm.GEE(y,z,groups=a.person_id.astype(str),family=sm.families.Binomial(),
                cov_struct=sm.cov_struct.Independence(),weights=w).fit(maxiter=200,ctol=1e-8)
        cov=np.asarray(fit.cov_params());eig=np.linalg.eigvalsh((cov+cov.T)/2)
        info.update(converged=bool(fit.converged),warnings=[str(x.message) for x in caught],
            robust_covariance_min_eigenvalue=float(eig.min()))
        if not fit.converged or not np.isfinite(fit.params).all() or not np.isfinite(cov).all() or eig.min()<=0:
            info['reason']='Convergence or finite positive-definite robust covariance requirement failed'
            results.append(info);return
        beta=float(fit.params['X_H']);se=float(fit.bse['X_H'])
        info.update(estimable=True,interaction_log_OR=beta,robust_se=se,
            interaction_OR=float(np.exp(beta)),interaction_OR_ci95=[float(np.exp(beta-1.96*se)),float(np.exp(beta+1.96*se))],
            p=float(2*norm.sf(abs(beta/se))),standardized=standardized(a,z,fit),
            coefficients={k:dict(beta=float(fit.params[k]),robust_se=float(fit.bse[k])) for k in z})
        fits[(d,endpoint,variant)]=(a,z,fit)
    except (ValueError,np.linalg.LinAlgError,FloatingPointError) as exc:
        info['reason']=type(exc).__name__+': '+str(exc)
    results.append(info)

# Four primary tests are computed and adjusted as one immutable family.
for d,df in frames.items():
    for endpoint in ['troponin_rise_12h','hospital_mortality']:
        fit_one(d,df,endpoint,'primary')
primary=[r for r in results if r['variant']=='primary'];assert len(primary)==4
adjusted=multipletests([r.get('p',1.0) for r in primary],alpha=.05,method='holm')[1]
for r,p in zip(primary,adjusted):r['holm_p']=float(p)
hj('primary_results.json',primary)
print(json.dumps([{k:r.get(k) for k in ['dataset','endpoint','n','events','estimable','interaction_OR','interaction_OR_ci95','holm_p','reason']} for r in primary],indent=2))

for d,df in frames.items():
    hs(d+' prespecified sensitivities')
    original=df.loc[df.original_hemoglobin.notna()].copy()
    original['hemoglobin']=original.original_hemoglobin
    original['hb_available_minute']=original.original_hb_available_minute
    timely=df.loc[df.hb_available_minute.notna()&(df.hb_available_minute<=0)].copy()
    for endpoint in ['troponin_rise_12h','hospital_mortality']:
        fit_one(d,original,endpoint,'explicit_id_only')
        fit_one(d,timely,endpoint,'entered_by_icu_admission')
        fit_one(d,df,endpoint,'without_care_covariates',care=False)

observation_results=[]
for d,df in frames.items():
    obs=df.troponin_rise_12h.notna().astype(int)
    folds=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=20260905)
    probabilities=pd.Series(np.nan,index=df.index);fold_audit=[]
    for fold,(train,test) in enumerate(folds.split(df,obs,groups=df.person_id)):
        a,b=df.iloc[train],df.iloc[test]
        assert not set(a.person_id)&set(b.person_id)
        z,med=design(a);v,_=design(b,medians=med)
        cols=[k for k in z if k!='intercept' and z[k].nunique()>1]
        scale=StandardScaler().fit(z[cols]);model=LogisticRegression(C=1,max_iter=2000,random_state=20260905)
        model.fit(scale.transform(z[cols]),obs.iloc[train]);probabilities.iloc[test]=model.predict_proba(scale.transform(v[cols]))[:,1]
        fold_audit.append(dict(fold=fold,train=len(train),test=len(test),person_disjoint=True,iterations=int(model.n_iter_.max())))
    assert probabilities.notna().all() and probabilities.between(0,1).all()
    weights=1/probabilities.clip(lower=.02)
    ess={}
    for x in [0,1]:
        w=weights.loc[(obs==1)&(df.exposed==x)];ess[str(x)]=float(w.sum()**2/(w*w).sum())
    bins=pd.qcut(probabilities,10,duplicates='drop')
    cal=pd.DataFrame(dict(predicted=probabilities,observed=obs,bin=bins)).groupby('bin',observed=True).agg(
        n=('observed','size'),predicted=('predicted','mean'),observed=('observed','mean')).reset_index(drop=True).to_dict('records')
    support=float((probabilities<.02).mean())<=.10 and min(ess.values())>=50
    w=weights.loc[obs==1]
    observation_results.append(dict(dataset=d,observed=int(obs.sum()),target=len(df),
        brier=float(brier_score_loss(obs,probabilities)),auc=float(roc_auc_score(obs,probabilities)),
        fraction_target_below_floor=float((probabilities<.02).mean()),ess_by_exposure=ess,
        ess_total=float(w.sum()**2/(w*w).sum()),weight_max=float(w.max()),support_pass=support,
        calibration=cal,folds=fold_audit,
        uncertainty='Robust outcome-model covariance conditional on fitted cross-fold observation weights; does not include weight estimation uncertainty'))
    saved=pd.DataFrame(dict(stay_id=df.stay_id,observation_probability=probabilities,observation_weight=weights))
    con.register('observation_frame',saved)
    con.execute(f'CREATE OR REPLACE TABLE {d}_observation_weights AS SELECT * FROM observation_frame')
    fit_one(d,df,'troponin_rise_12h','observation_weighted',weights=weights)
hj('observation_weighting.json',observation_results)
hj('all_model_results.json',results)
replicated=[]
for endpoint in ['troponin_rise_12h','hospital_mortality']:
    family=[r for r in primary if r['endpoint']==endpoint]
    if all(r.get('estimable') and r.get('interaction_OR',0)>1 and r.get('holm_p',1)<.05 for r in family):
        replicated.append(endpoint)
hj('analysis_manifest.json',dict(protocol_sha256=HB_INTERACTION_PROTOCOL_SHA256,
    environment='google_colab' if IN_COLAB else 'local',completed_utc=datetime.now(timezone.utc).isoformat(),
    primary_tests=4,positive_statistically_replicated_endpoints=replicated,
    biological_discovery=False,goal_complete=False,
    model_code_sha256=hashlib.sha256((PROJECT/'scripts/hemoglobin_interaction_cells.py').read_bytes()).hexdigest()))
for name,info in fingerprints.items():
    s=Path(info['path']).stat();assert s.st_size==info['bytes'] and s.st_mtime_ns==info['mtime_ns'],name
con.close()
hs('frozen hemoglobin interaction analysis','complete')

frozen hemoglobin interaction analysis running
[
  {
    "dataset": "eicu",
    "endpoint": "troponin_rise_12h",
    "n": 659,
    "events": 159,
    "estimable": true,
    "interaction_OR": 0.8397431599793778,
    "interaction_OR_ci95": [
      0.5853281888989109,
      1.204740499613862
    ],
    "holm_p": 1.0,
    "reason": null
  },
  {
    "dataset": "eicu",
    "endpoint": "hospital_mortality",
    "n": 3945,
    "events": 474,
    "estimable": true,
    "interaction_OR": 1.068569273984573,
    "interaction_OR_ci95": [
      0.8962922430891,
      1.2739598073152212
    ],
    "holm_p": 1.0,
    "reason": null
  },
  {
    "dataset": "mimic",
    "endpoint": "troponin_rise_12h",
    "n": 516,
    "events": 145,
    "estimable": true,
    "interaction_OR": 0.9736437374812216,
    "interaction_OR_ci95": [
      0.6709395327309153,
      1.412917381209964
    ],
    "holm_p": 1.0,
    "reason": null
  },
  {
    "dataset": "mimic",
    "endpoint": "hospital_mortality",
    "n": 345

In [23]:
# Independent matrix reconstruction of the primary patient-cluster covariance.
# This checks the real fitted data, not fabricated patient examples.
checks=[]
for (d,endpoint,variant),(a,z,fit) in fits.items():
    if variant!='primary':continue
    av=z.to_numpy();y=a[endpoint].to_numpy();p=expit(av@np.asarray(fit.params))
    independent=sm.GLM(y,av,family=sm.families.Binomial()).fit(maxiter=200,tol=1e-10)
    score=pd.DataFrame(av*(y-p)[:,None],index=a.index)
    score['person']=a.person_id.astype(str).to_numpy()
    cluster_scores=score.groupby('person').sum().to_numpy()
    bread=np.linalg.inv(av.T@(av*(p*(1-p))[:,None]))
    covariance=bread@(cluster_scores.T@cluster_scores)@bread
    beta_error=float(np.max(np.abs(np.asarray(independent.params)-np.asarray(fit.params))))
    cov_error=float(np.max(np.abs(covariance-np.asarray(fit.cov_params()))))
    assert np.allclose(independent.params,fit.params,atol=1e-5,rtol=1e-5),(d,endpoint,beta_error)
    assert np.allclose(covariance,fit.cov_params(),atol=1e-8,rtol=1e-5),(d,endpoint,cov_error)
    checks.append(dict(dataset=d,endpoint=endpoint,independent_GLM_max_beta_error=beta_error,
        manually_reconstructed_cluster_covariance_max_error=cov_error,passed=True))
hj('independent_model_validation.json',dict(primary_model_checks=checks,
    all_estimable_primary_models_checked=len(checks)==sum(r['estimable'] for r in primary),
    patient_level_outputs_private=True,environment='google_colab' if IN_COLAB else 'local'))
print(json.dumps(checks,indent=2))

AssertionError: ('eicu', 'hospital_mortality', 2.00000000188955)

In [ ]:
primary=json.loads((OUT/'primary_results.json').read_text())
all_results=json.loads((OUT/'all_model_results.json').read_text())
weights=json.loads((OUT/'observation_weighting.json').read_text())
manifest=json.loads((OUT/'analysis_manifest.json').read_text())
lines=['# Prespecified hemoglobin interaction results','',
    '**Actual execution: '+manifest['environment']+'. No established biological discovery or mortality-reduction finding.**','',
    'Only the existing MIMIC/eICU HF cohorts and SpO2 exposure were used. The protocol was frozen before opening Hb-by-outcome associations. Hb came from the pre-ICU CBC selection with the validated linkage-only amendment.','',
    'The interaction is the ratio of SpO2-exposure **odds ratios per 2 g/dL lower hemoglobin**, adjusted for absolute oxygenation, hypoxemic burden, sampling, physiology and documented treatment. It is not a risk ratio or a transfusion effect.','',
    '| Database | Endpoint | N | Events | Interaction OR (95% CI) | Holm p |','|---|---|---:|---:|---|---:|']
for r in primary:
    effect=(f"{r['interaction_OR']:.3f} ({r['interaction_OR_ci95'][0]:.3f}–{r['interaction_OR_ci95'][1]:.3f})" if r['estimable'] else 'Not estimable')
    lines.append(f"| {r['dataset']} | {r['endpoint']} | {r['n']} | {r['events']} | {effect} | {r['holm_p']:.4g} |")
lines+=['','Holm correction covers all four source/endpoint primary tests. Both databases were previously explored and are not untouched validation samples.','',
    '## Prespecified sensitivities','',
    '| Database | Endpoint | Analysis | N | Interaction OR (95% CI) |','|---|---|---|---:|---|']
for r in all_results:
    if r['variant']=='primary':continue
    effect=(f"{r['interaction_OR']:.3f} ({r['interaction_OR_ci95'][0]:.3f}–{r['interaction_OR_ci95'][1]:.3f})" if r['estimable'] else r.get('reason','Not estimable'))
    lines.append(f"| {r['dataset']} | {r['endpoint']} | {r['variant']} | {r['n']} | {effect} |")
lines+=['','These are sensitivity analyses, not additional confirmatory tests. The weighted analysis assumes troponin ascertainment is explainable by measured covariates. Its reported covariance conditions on estimated weights.','']
for w in weights:
    lines.append(f"- {w['dataset']}: {w['fraction_target_below_floor']:.1%} of observation probabilities below 0.02; weighted ESS {w['ess_total']:.1f}, unexposed/exposed {w['ess_by_exposure']['0']:.1f}/{w['ess_by_exposure']['1']:.1f}; frozen weight-support gate {'passed' if w['support_pass'] else 'failed'}.")
lines+=['','## Absolute mortality contrasts','',
    'These standardize conditional model predictions at Hb 8 and 12 g/dL over the same observed covariate distribution. They are associative contrasts, not the effect of changing Hb.','']
for r in primary:
    if r['endpoint']!='hospital_mortality' or not r['estimable']:continue
    s=r['standardized']
    if not s['support_pass']:
        lines.append(f"- {r['dataset']}: frozen Hb-neighborhood support gate failed; no standardized contrast reported.")
        continue
    z=s['difference_in_differences'];rd=s['contrasts']
    lines.append(f"- {r['dataset']}: exposure risk difference at Hb 8 = {100*rd['RD_hb8']['value']:.2f} percentage points; at Hb 12 = {100*rd['RD_hb12']['value']:.2f}. Difference in differences {100*z['value']:.2f} points (95% CI {100*z['ci95'][0]:.2f} to {100*z['ci95'][1]:.2f}).")
lines+=['','## Decision and limits','']
if manifest['positive_statistically_replicated_endpoints']:
    lines.append('A positive interaction passed the four-test statistical replication screen for: '+', '.join(manifest['positive_statistically_replicated_endpoints'])+'. This remains a candidate requiring the prespecified effect-size and sensitivity assessment; it is not a completed discovery.')
else:
    lines.append('No endpoint passed the prespecified positive, multiplicity-adjusted replication requirement across both databases. This candidate does not establish that low hemoglobin amplifies the existing SpO2 association. Wide intervals remain inconclusive rather than proving the absence of an effect.')
lines+=['',
    'Incomplete eICU pressure/treatment documentation, selective Hb/troponin testing, inferred MIMIC admission links, bleeding/transfusion confounding, and care variables measured during the exposure window limit mechanistic interpretation. Revision/store times do not uniformly identify first clinical availability. The 12-hour troponin endpoint indicates assay-matched rise, not adjudicated MI; hospital mortality does not measure a fixed-time treatment response.','',
    'The broad anemia–hypoxemia hypothesis is prior knowledge: [COVID-19 interaction study](https://pmc.ncbi.nlm.nih.gov/articles/PMC9447453/) and [MINT HF analysis](https://pmc.ncbi.nlm.nih.gov/articles/PMC11999761/). Neither supplies patient data to this experiment.','',
    'Primary GEE estimates were checked against a separately fitted GLM, and robust covariance was independently reconstructed from patient-level scores. Aggregate evidence is in research/spo2_hemoglobin_interaction; patient records remain in private Drive Data. The runnable workflow is PhysioGraph_Biological_Discovery.ipynb. The overall biological-discovery goal remains incomplete.','']
report='\n'.join(lines)
(PROJECT/'docs/SPO2_HEMOGLOBIN_INTERACTION_RESULTS.md').write_text(report)
print(report)